Problem 1 – Row-wise ℓ₂ normalization
-------------------------------------
Given a float tensor **X** of shape `(B, D)`, return a tensor **Y**
with the same shape such that every row of **Y** has (approximate)
ℓ₂-norm == 1.

Mathematically  
 Yᵢ = Xᵢ / (‖Xᵢ‖₂ + ε)

Constraints
-----------
* 1 ≤ B·D ≤ 10⁷
* Use **no Python for-loops**.

Example
-------
```python
X = torch.tensor([[3., 4.]])
Y = normalize_rows(X)
Y.norm(dim=1)

>>> tensor([1.])
```

In [4]:
import torch

def normalize_rows(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return X / (torch.linalg.vector_norm(X, ord=2, dim=-1, keepdim=True) + eps)


# ---------- reference + tests (do not modify) ----------
def _ref_normalize_rows(X, eps=1e-8):
    return X / (X.norm(dim=1, keepdim=True) + eps)

def _self_check():
    X = torch.randn(5, 7)
    try:
        Y = normalize_rows(X)
    except NotImplementedError:
        print("🔧  Implement normalize_rows and re-run the cell.")
        return
    assert Y.shape == X.shape, "Shape mismatch"
    assert torch.allclose(Y, _ref_normalize_rows(X)), "Wrong answer"
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 2 – Channel-wise z-score
--------------------------------
For an image batch tensor **imgs** of shape `(N, C, H, W)` compute

    (img − μ_c) / (σ_c + ε)

where μ_c and σ_c are the mean and std of channel *c* over the
(N, H, W) axes.

Return a tensor of the same shape.

No Python loops allowed.

In [10]:
import torch

def channel_zscore(imgs: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    N, C, H, W = imgs.shape
    tmp_imgs = imgs.permute(1, 0, 2, 3).reshape(C, N*H*W)
    avg = tmp_imgs.mean(dim=-1).reshape(1, C, 1, 1)
    std = tmp_imgs.std(dim=-1).reshape(1, C, 1, 1)
    return (imgs - avg) / (std + eps)

# reference & test
def _ref_channel_zscore(imgs, eps=1e-5):
    mean = imgs.mean(dim=(0, 2, 3), keepdim=True)
    std  = imgs.std (dim=(0, 2, 3), keepdim=True) + eps
    return (imgs - mean) / std

def _self_check():
    imgs = torch.randn(4, 3, 16, 16)
    try:
        out = channel_zscore(imgs)
    except NotImplementedError:
        print("🔧  Implement channel_zscore.")
        return
    assert torch.allclose(out, _ref_channel_zscore(imgs), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 3 – Numerically stable softmax per row
----------------------------------------------
Given a tensor **S** of shape `(B, K)`, return

    softmax(Sᵢ)  for each row i,

implemented with the usual “shift by max” trick to avoid overflow.

In [13]:
import torch

def stable_row_softmax(S: torch.Tensor) -> torch.Tensor:
    B, K = S.shape
    S = S - S.max(dim=-1, keepdim=True).values
    return S.exp() / S.exp().sum(dim=-1, keepdim=True)


def _ref_stable_row_softmax(S):
    Z = S - S.max(dim=1, keepdim=True).values
    eZ = Z.exp()
    return eZ / eZ.sum(dim=1, keepdim=True)

def _self_check():
    S = torch.randn(3, 9) * 25
    try:
        out = stable_row_softmax(S)
    except NotImplementedError:
        print("🔧  Implement stable_row_softmax.")
        return
    assert torch.allclose(out, _ref_stable_row_softmax(S), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 4 – Masked fill
-----------------------
Given
* **logits** of arbitrary shape `(*)`
* a boolean **mask** of the same shape  
return a tensor where entries with `mask == False` are replaced by
−∞ (so they will become zero after softmax).

In [14]:
import torch

def masked_logits(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill_(~mask, -torch.inf)


def _ref_masked_logits(logits, mask):
    return logits.masked_fill(~mask, float("-inf"))

def _self_check():
    L = torch.randn(2, 5)
    M = torch.rand(2, 5) > .4
    try:
        out = masked_logits(L, M)
    except NotImplementedError:
        print("🔧  Implement masked_logits.")
        return
    assert torch.isneginf(out[~M]).all()
    assert torch.equal(out, _ref_masked_logits(L, M))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 5 – Batched cosine
--------------------------
Input
* **A** ∈ ℝ^{B×M×D}
* **B** ∈ ℝ^{B×N×D}

Output **S** ∈ ℝ^{B×M×N} where  
 S[b, i, j] = cosine_similarity(A[b,i], B[b,j]).

In [15]:
import torch

def batched_cosine(A: torch.Tensor, B: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    b, m, d = A.shape
    b, n, d = B.shape

    P = torch.einsum("bmd,bnd->bmn", A, B)
    normA = torch.linalg.vector_norm(A, dim=-1, keepdim=True)
    normB = torch.linalg.vector_norm(B, dim=-1, keepdim=True).view(b, 1, n)

    return P / (normA * normB + eps)


def _ref_batched_cosine(A, B, eps=1e-8):
    A = A / (A.norm(dim=-1, keepdim=True) + eps)
    B = B / (B.norm(dim=-1, keepdim=True) + eps)
    return torch.einsum("bmd,bnd->bmn", A, B)

def _self_check():
    A, B = torch.randn(2, 3, 4), torch.randn(2, 5, 4)
    try:
        out = batched_cosine(A, B)
    except NotImplementedError:
        print("🔧  Implement batched_cosine.")
        return
    assert torch.allclose(out, _ref_batched_cosine(A, B), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 6 – One-hot
-------------------
Given an integer index tensor **idx** of shape `(B,)` and
num_classes =C, return a one-hot tensor `(B, C)` without any loop.

In [17]:
import torch

def one_hot(idx: torch.Tensor, num_classes: int) -> torch.Tensor:
    B, = idx.shape
    idx = idx.view(B, 1)
    val = torch.ones(B, 1)
    res = torch.zeros(B, num_classes)
    res.scatter_(1, idx, val)
    return res


def _ref_one_hot(idx, C):
    out = torch.zeros(idx.size(0), C, device=idx.device)
    out.scatter_(1, idx.unsqueeze(1), 1.0)
    return out

def _self_check():
    i = torch.tensor([2, 0, 1])
    try:
        out = one_hot(i, 4)
    except NotImplementedError:
        print("🔧  Implement one_hot.")
        return
    assert torch.equal(out, _ref_one_hot(i, 4))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 7 – Masked cumulative product
-------------------------------------
Input
* **X** ∈ ℝ^{B×T} (positive)
* **reset** ∈ {0,1}^{B×T}

For each row, compute the running product but **restart at 1** whenever
reset==1 at that timestep.

In [21]:
import torch

def masked_cumprod(X: torch.Tensor, reset: torch.Tensor) -> torch.Tensor:
    assert torch.all(X != 0.0), "assumption voilated"
    
    P = torch.cumprod(X, dim=1)
    PP = torch.cat((torch.ones_like(P[:, :1]), P[:, :-1]), dim=1)

    index = reset.cumsum(dim=1)
    tmp = torch.zeros_like(X)
    tmp.scatter_reduce_(1, index, reset * PP, reduce="sum")
    tmp[:, 0] = 1

    baseline = torch.gather(tmp, 1, index)
    return P / baseline

def _ref_masked_cumprod(X, reset):
    B, T = X.shape
    running = torch.ones(B)
    res = torch.ones(B, T)
    for t in range(T):
        running = torch.where(reset[:, t], X[:, t], running * X[:, t])
        res[:, t] = running
    return res
    
def _self_check():
    X = torch.tensor([[2.,3.,4.,5.],[1.,2.,3.,4.]])
    R = torch.tensor([[0,1,0,0],[0,0,1,0]], dtype=torch.bool)
    try:
        out = masked_cumprod(X, R)
    except NotImplementedError:
        print("🔧  Implement masked_cumprod.")
        return
    assert torch.allclose(out, _ref_masked_cumprod(X, R))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 8 – Conv2d (no bias) with unfold
----------------------------------------
Re-implement `torch.nn.functional.conv2d` for square kernels
using `torch.nn.functional.unfold`.

Signature
---------
conv2d_im2col(x, w, stride=1, padding=0)

In [1]:
import torch

def conv2d_im2col(x: torch.Tensor, w: torch.Tensor,
                  stride: int = 1, padding: int = 0) -> torch.Tensor:
    B, Ci, H, W = x.shape
    Co, _, k, _ = w.shape
    x_unf = torch.nn.functional.unfold(x, k, padding=padding, stride=stride) # B x (Ci * k * k) x L
    w = w.view(Co, Ci * k * k)
    out = torch.einsum("bdl,cd->bcl", x_unf, w)
    Ho = (H + 2*padding - k) // stride + 1
    Wo = (W + 2*padding - k) // stride + 1
    return out.view(B, Co, Ho, Wo)

def _ref_conv2d_im2col(x, w, stride=1, padding=0):
    N, C_in, H, W = x.shape
    C_out, _, k, _ = w.shape
    x_unf = torch.nn.functional.unfold(x, k, padding=padding, stride=stride)
    out = w.view(C_out, -1) @ x_unf
    H_out = (H + 2*padding - k)//stride + 1
    W_out = (W + 2*padding - k)//stride + 1
    return out.view(N, C_out, H_out, W_out)

def _self_check():
    x = torch.randn(1,3,5,5); w = torch.randn(4,3,3,3)
    try:
        out = conv2d_im2col(x, w, padding=1)
    except NotImplementedError:
        print("🔧  Implement conv2d_im2col.")
        return
    assert torch.allclose(out, _ref_conv2d_im2col(x, w, padding=1), atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 9 – Batched k-NN (squared Euclidean)
--------------------------------------------
Given X ∈ ℝ^{B×N×D}, Y ∈ ℝ^{B×M×D} and k, return indices of the k
closest Y-vectors for every X-vector within each batch.

In [5]:
import torch

def batched_knn(X: torch.Tensor, Y: torch.Tensor, k: int) -> torch.Tensor:
    B, N, D = X.shape
    B, M, D = Y.shape

    xx = (X * X).sum(dim=-1, keepdim=True)                  # B, N, 1
    yy = (Y * Y).sum(dim=-1, keepdim=True).permute(0, 2, 1) # B, 1, M
    xy = torch.einsum("bnd,bmd->bnm", X, Y)                 # B, N, M

    l2 = xx + yy - 2*xy                                     # B, N, M
    _, idx = l2.topk(k, dim=-1, largest=False)
    return idx


def _ref_batched_knn(X, Y, k):
    d = (X.pow(2).sum(-1, keepdim=True)
         + Y.pow(2).sum(-1).unsqueeze(1)
         - 2*torch.einsum("bnd,bmd->bnm", X, Y))
    _, idx = d.topk(k, dim=-1, largest=False)
    return idx

def _self_check():
    X,Y = torch.randn(2,4,6), torch.randn(2,5,6)
    try:
        idx = batched_knn(X, Y, 3)
    except NotImplementedError:
        print("🔧  Implement batched_knn.")
        return
    assert torch.equal(idx, _ref_batched_knn(X,Y,3))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 10 – Brevity penalty
----------------------------
For sentence lengths `len_c, len_r` (candidate vs reference),
return

    BP = 1                   if len_c > len_r  
         exp(1 − len_r/len_c) otherwise

Vectorized over a batch of shape `(B,)`.

In [6]:
import torch
import math

def brevity_penalty(len_c: torch.Tensor, len_r: torch.Tensor) -> torch.Tensor:
    len_c, len_r = len_c.float(), len_r.float()
    return torch.where(len_c > len_r, torch.ones_like(len_c), torch.exp(1 - len_r / (len_c + 1e-8)))


def _ref_bp(lc, lr):
    lc, lr = lc.float(), lr.float()
    return torch.where(lc > lr, torch.ones_like(lc), torch.exp(1 - lr / (lc + 1e-8)))

def _self_check():
    lc, lr = torch.tensor([10, 5]), torch.tensor([8, 9])
    try:
        bp = brevity_penalty(lc, lr)
    except NotImplementedError:
        print("🔧  Implement brevity_penalty.")
        return
    assert torch.allclose(bp, _ref_bp(lc, lr))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 11 – One-step GRU
-------------------------
The GRU update consists of the following steps:

1. Reset gate:
    r = sigmoid(Wx_r @ x + Wh_r @ h + b_r)

2. Update gate:
    z = sigmoid(Wx_z @ x + Wh_z @ h + b_z)

3. Candidate (new) state:
    n = tanh(Wx_n @ x + r ⊙ (Wh_n @ h) + b_n)

4. Final hidden state:
    h_next = (1 - z) ⊙ h + z ⊙ n

In [11]:
import torch

def gru_cell(x: torch.Tensor, h: torch.Tensor,
             Wx: torch.Tensor, Wh: torch.Tensor,
             b: torch.Tensor) -> torch.Tensor:
    #  x:  B, D
    #  h:  B, H
    # Wx: 3H, D
    # Wh: 3H, H
    #  b: 3H
    B, D = x.shape
    _, H = h.shape

    x_proj = x @ Wx.T
    h_proj = h @ Wh.T
    b_proj = b

    x_r, x_z, x_n = x_proj.chunk(3, dim=-1)
    h_r, h_z, h_n = h_proj.chunk(3, dim=-1)
    b_r, b_z, b_n = b_proj.chunk(3, dim=-1)

    r = torch.sigmoid(x_r + h_r + b_r)
    z = torch.sigmoid(x_z + h_z + b_z)
    n = torch.tanh(x_n + r * h_n + b_n)

    return (1 - z) * h + z * n
    


def _ref_gru_cell(x,h,Wx,Wh,b):
    H = h.size(1)

    # Input and hidden projections for all 3 gates
    x_proj = x @ Wx.T + b      # (B, 3H)
    h_proj = h @ Wh.T          # (B, 3H)

    # Split into reset, update, and new gate
    x_r, x_z, x_n = x_proj.chunk(3, dim=-1)
    h_r, h_z, h_n = h_proj.chunk(3, dim=-1)

    # Gates
    r = torch.sigmoid(x_r + h_r)             # Reset gate
    z = torch.sigmoid(x_z + h_z)             # Update gate
    n = torch.tanh(x_n + r * h_n)            # New gate with r ⊙ h_n

    # Final hidden state
    h_next = (1 - z) * h + z * n             # GRU interpolation
    return h_next

def _self_check():
    B,D,H=2,3,4
    args=[torch.randn(B,D),torch.randn(B,H),
          torch.randn(3*H,D),torch.randn(3*H,H),
          torch.randn(3*H)]
    try:
        out=gru_cell(*args)
    except NotImplementedError:
        print("🔧  Implement gru_cell.")
        return
    assert torch.allclose(out,_ref_gru_cell(*args),atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 12 – Causal self-attention scores
-----------------------------------------
Given Q,K ∈ ℝ^{B×T×d}, return the softmaxed score matrix
`(B, T, T)` with an upper-triangular causal mask.

In [20]:
import torch, math

def causal_attention_scores(Q: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    # Q: B, T, D
    # K: B, S, D
    B, T, D = Q.shape
    _, S, _ = K.shape
    
    mask = torch.tril(torch.ones(T, S, dtype=torch.bool), diagonal=0)
    
    dot_products = torch.einsum("btd,bsd->bts", Q, K) / math.sqrt(D)
    dot_products.masked_fill_(~mask, -torch.inf)

    attn_weights = torch.softmax(dot_products, dim=-1)
    return attn_weights
    

def _ref_causal(Q,K):
    d = Q.size(-1)
    S = torch.einsum("btd,bkd->btk",Q,K)/math.sqrt(d)
    T = S.size(-1)

    mask = torch.triu(torch.ones(T,T,device=Q.device,dtype=torch.bool),1)
    return torch.softmax(S.masked_fill(mask,float("-inf")), -1)

def _self_check():
    Q,K=torch.randn(1,6,4),torch.randn(1,6,4)
    try: out=causal_attention_scores(Q,K)
    except NotImplementedError:
        print("🔧  Implement causal_attention_scores.")
        return
    assert torch.allclose(out,_ref_causal(Q,K),atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 13 – Min–max scale every row
------------------------------------
For X ∈ ℝ^{B×D} return

    Yᵢⱼ = (Xᵢⱼ − min(Xᵢ)) / (max(Xᵢ) − min(Xᵢ) + ε)

so every row is mapped to the range [0, 1].

No Python loops allowed.

In [22]:
import torch

def minmax_rows(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    # X: (B, D)
    Xmin, _ = torch.min(X, dim=-1, keepdim=True)
    Xmax, _ = torch.max(X, dim=-1, keepdim=True)

    Y = (X - Xmin) / (Xmax - Xmin + eps)
    return Y


# reference & test
def _ref_minmax_rows(X, eps=1e-8):
    xmin = X.min(dim=1, keepdim=True).values
    xmax = X.max(dim=1, keepdim=True).values
    return (X - xmin) / (xmax - xmin + eps)

def _self_check():
    X = torch.randn(4, 7)
    try:
        out = minmax_rows(X)
    except NotImplementedError:
        print("🔧  Implement minmax_rows.")
        return
    assert torch.allclose(out, _ref_minmax_rows(X), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 14 – Batch RGB→gray conversion
--------------------------------------
Given imgs ∈ ℝ^{N×3×H×W} with values in [0,1], return a grayscale
tensor of shape (N, 1, H, W) using weights

    Y = 0.299 R + 0.587 G + 0.114 B

In [23]:
import torch

def rgb_to_gray(imgs: torch.Tensor) -> torch.Tensor:
    # imgs: B, C, H, W
    w = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
    return (imgs * w).sum(dim=1, keepdim=True)


def _ref_rgb_to_gray(imgs):
    w = torch.tensor([0.299, 0.587, 0.114], device=imgs.device).view(1,3,1,1)
    return (imgs * w).sum(1, keepdim=True)

def _self_check():
    imgs = torch.rand(2,3,8,8)
    try:
        out = rgb_to_gray(imgs)
    except NotImplementedError:
        print("🔧  Implement rgb_to_gray.")
        return
    assert torch.allclose(out, _ref_rgb_to_gray(imgs), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 15 – σ(z) and σ'(z)
---------------------------
Return both the logistic sigmoid and its derivative for an
arbitrary-shaped tensor z (no loops).

In [24]:
import torch

def sigmoid_and_grad(z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    s = torch.sigmoid(z)
    return s, s*(1-s)

def _ref_sig_and_grad(z):
    s = torch.sigmoid(z)
    return s, s*(1-s)

def _self_check():
    z = torch.randn(5,3,2)
    try:
        s, ds = sigmoid_and_grad(z)
    except NotImplementedError:
        print("🔧  Implement sigmoid_and_grad.")
        return
    s_ref, ds_ref = _ref_sig_and_grad(z)
    assert torch.allclose(s, s_ref) and torch.allclose(ds, ds_ref)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 16 – Trace & diagonal per batch
---------------------------------------
For M ∈ ℝ^{B×D×D} return

    trace  – shape (B,)
    diag   – shape (B,D)

using only tensor ops.

In [25]:
import torch

def batch_trace_diag(M: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    # M: (B, D, D)
    diag = M.diagonal(dim1=1, dim2=2)
    trace = diag.sum(dim=-1)
    return trace, diag


def _ref_batch_trace_diag(M):
    diag = M.diagonal(dim1=-2, dim2=-1)
    return diag.sum(-1), diag

def _self_check():
    M = torch.randn(3,4,4)
    try:
        tr, d = batch_trace_diag(M)
    except NotImplementedError:
        print("🔧  Implement batch_trace_diag.")
        return
    tr_ref, d_ref = _ref_batch_trace_diag(M)
    assert torch.allclose(tr, tr_ref) and torch.allclose(d, d_ref)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 17 – Label-smoothed CE loss
-----------------------------------
Given logits (B,C), integer targets (B,) and smoothing α ∈ [0,1),
return the scalar loss

    L = − mean_i  sum_c  q_ic  log softmax_i,c

where q is the smoothed one-hot:  
 q_i,target = 1−α, others = α/(C−1).

No loops.

In [26]:
import torch, torch.nn.functional as F

def ls_ce_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float) -> torch.Tensor:
    B, C = logits.shape
    q = torch.ones_like(logits) * (alpha / (C-1)) # B, C
    q.scatter_(1, targets.view(B, 1), 1-alpha)    # B, C
    return -(q * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

def _ref_ls_ce(logits, targets, alpha):
    B,C = logits.shape
    q = torch.full((B,C), alpha/(C-1), device=logits.device)
    q.scatter_(1, targets.unsqueeze(1), 1-alpha)
    return -(q * F.log_softmax(logits,1)).sum(1).mean()

def _self_check():
    logit = torch.randn(6,10)
    tgt   = torch.randint(0,10,(6,))
    try:
        loss = ls_ce_loss(logit, tgt, 0.1)
    except NotImplementedError:
        print("🔧  Implement ls_ce_loss.")
        return
    assert torch.allclose(loss, _ref_ls_ce(logit,tgt,0.1), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 18 – Jaccard indices
----------------------------
A, B ∈ {0,1}^{B×N×D}.  
Compute J[b, i] = |A[b,0] ∩ B[b,i]| / |A[b,0] ∪ B[b,i]|.

Return J shape (B, N).

In [27]:
import torch

def jaccard_pair(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    A0 = A[:, :1, :] # B, 1, D
    inter = (A0 & B).sum(-1)
    union = (A0 | B).sum(-1)
    return inter / (union + 1e-8)


def _ref_jaccard_pair(A,B):
    inter = (A[:,:1] & B).sum(-1)
    union = (A[:,:1] | B).sum(-1)
    return inter / (union + 1e-8)

def _self_check():
    A = (torch.rand(2,1,5)>.5).bool().repeat(1,3,1)
    B = (torch.rand(2,3,5)>.5).bool()
    try:
        out = jaccard_pair(A,B)
    except NotImplementedError:
        print("🔧  Implement jaccard_pair.")
        return
    assert torch.allclose(out, _ref_jaccard_pair(A,B), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 19 – Dice coefficient (mean over classes)
-------------------------------------------------
pred, mask ∈ {0,1}^{N×C×H×W}.  
Return the mean Dice over the C classes:

    Dice_c = (2|∩| + ε) / (|pred_c| + |mask_c| + ε)

In [28]:
import torch

def dice_coeff(pred: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    inter = (pred & mask).sum((0,2,3))
    union = pred.sum((0,2,3)) + mask.sum((0,2,3))
    return ((2*inter+eps)/(union+eps)).mean()


def _ref_dice(pred, mask, eps=1e-6):
    inter = (pred & mask).sum((0,2,3))
    union = pred.sum((0,2,3)) + mask.sum((0,2,3))
    return ((2*inter+eps)/(union+eps)).mean()

def _self_check():
    p = (torch.rand(4,3,10,10)>.5)
    m = (torch.rand(4,3,10,10)>.5)
    try:
        d = dice_coeff(p,m)
    except NotImplementedError:
        print("🔧  Implement dice_coeff.")
        return
    assert torch.allclose(d, _ref_dice(p,m), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 20 – Causal conv1d (no bias)
------------------------------------
x: (B,C,T), w: (C_out,C,k).  
Pad (k−1) zeros on the *left* so output has length T.
No loops; use `unfold` or `tensor.unfold`.

In [29]:
import torch

def causal_conv1d(x: torch.Tensor, w: torch.Tensor, dilation: int = 1) -> torch.Tensor:
    B, C_in, T = x.shape
    C_out, C_in, k = w.shape

    x_pad = torch.nn.functional.pad(x, (k-1,0), "constant", 0)
    x_unf = x_pad.unfold(2,k,1)                                 # B, C_in, T, k
    return torch.einsum("bctk,ock->bot", x_unf, w)

def _ref_causal_conv1d(x,w,d=1):
    k = w.size(-1)
    pad = (k-1)*d
    xpad = torch.nn.functional.pad(x,(pad,0))
    x_unf = xpad.unfold(2,k,1).transpose(1,2)  # (B,T,C,k)
    return torch.einsum("btck,ock->bot", x_unf, w)

def _self_check():
    x = torch.randn(1,2,6)
    w = torch.randn(3,2,3)
    try:
        out = causal_conv1d(x,w)
    except NotImplementedError:
        print("🔧  Implement causal_conv1d.")
        return
    assert torch.allclose(out, _ref_causal_conv1d(x,w), atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 21 – Rolling mean
-------------------------
For x ∈ ℝ^{B×T} and window w (w≤T) return
shape (B, T−w+1) where each entry is the mean of a sliding length-w
window.  Use tensor.unfold; no loops.

In [31]:
import torch

def rolling_mean(x: torch.Tensor, window: int) -> torch.Tensor:
    B, T = x.shape
    x_unf = x.unfold(1,window,1).mean(-1)
    return x_unf

def _ref_rolling_mean(x, w):
    return x.unfold(1,w,1).mean(-1)

def _self_check():
    a = torch.arange(6).float().view(1,-1)
    try:
        out = rolling_mean(a,3)
    except NotImplementedError:
        print("🔧  Implement rolling_mean.")
        return
    assert torch.allclose(out, _ref_rolling_mean(a,3))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 22 – Gather (B, T, D) at time indices
---------------------------------------------
seq: (B,T,D), idx: (B,K) → out: (B,K,D) using `tensor.gather`.

In [32]:
import torch

def batch_gather(seq: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    B, T, D = seq.shape
    B, K = idx.shape
    idx = idx.view(B, K, 1).expand(B, K, D)
    return torch.gather(seq, 1, idx)


def _ref_batch_gather(seq, idx):
    return seq.gather(1, idx.unsqueeze(-1).expand(-1,-1,seq.size(-1)))

def _self_check():
    s = torch.randn(2,5,4)
    i = torch.tensor([[1,3],[0,4]])
    try:
        out = batch_gather(s,i)
    except NotImplementedError:
        print("🔧  Implement batch_gather.")
        return
    assert torch.allclose(out, _ref_batch_gather(s,i))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 23 – Sparsemax
----------------------
Implement sparsemax (Martins & Astudillo 2016) along a given
dimension dim (default −1).  See paper for algorithm.

In [9]:
import torch

def sparsemax(z: torch.Tensor, dim: int = -1) -> torch.Tensor:
    z_sorted, _ = torch.sort(z, dim=dim, descending=True)
    z_cumsum = z_sorted.cumsum(dim)
    
    k = torch.arange(1, z.size(dim)+1)
    k_shape = [1] * z.dim()
    k_shape[dim] = -1
    k = k.view(*k_shape)
    
    support = (z_cumsum - k * z_sorted) < 1
        
    tau = ((z_sorted * support).sum(dim=dim, keepdim=True) - 1) / support.sum(dim=dim, keepdim=True)

    return torch.clamp(z - tau, min=0)
    

def _ref_sparsemax(z, dim=-1):
    # 1) sort z in descending order
    z_sorted, _ = torch.sort(z, dim=dim, descending=True)

    # 2) compute running sums of sorted z, subtract 1
    z_cumsum = z_sorted.cumsum(dim) - 1

    # 3) create the 1-based index k for each position along `dim`
    r = torch.arange(1, z.size(dim) + 1, device=z.device, dtype=z.dtype)
    # reshape so it can broadcast along every other dimension
    r_shape = [1] * z.dim()
    r_shape[dim] = -1
    r = r.view(*r_shape)

    # 4) find the support: z_k > tau_k
    support = z_sorted > z_cumsum / r
    k_support = support.sum(dim=dim, keepdim=True)

    # 5) compute the threshold τ
    tau = ( (z_sorted * support).sum(dim=dim, keepdim=True) - 1 ) / k_support

    # 6) project onto the simplex
    return torch.clamp(z - tau, min=0.0)

def _self_check():
    z = torch.randn(3,7)
    try:
        out = sparsemax(z)
    except NotImplementedError:
        print("🔧  Implement sparsemax.")
        return
    assert torch.allclose(out, _ref_sparsemax(z), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 24 – Multi-head causal self-attention (no bias)
------------------------------------------------------
Q,K,V ∈ ℝ^{B×T×H×d}.  
Return context: (B,T,H,d) using dot-product attention with an
upper-triangular causal mask (each position attends to ≤ its past).
No loops.

In [12]:
import torch, math

def mh_causal_attn(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    # Q: B, T, H, D
    # K: B, S, H, D
    # V: B, S, H, D

    B, T, H, D = Q.shape

    scores = torch.einsum("bthd,bshd->bhts", Q, K) / math.sqrt(D)
    masks = torch.tril(torch.ones(T, T, dtype=torch.bool), diagonal=0).view(1, 1, T, T)
    scores.masked_fill_(~masks, -torch.inf)
    weights = torch.softmax(scores, dim=-1)
    return torch.einsum("bhts,bshd->bthd", weights, V)
    

def _ref_mh_attn(Q,K,V):
    d = Q.size(-1)
    scores = torch.einsum("bthd,bshd->bhts", Q, K)/math.sqrt(d)
    T = scores.size(-1)
    mask = torch.triu(torch.ones(T,T,device=Q.device,dtype=torch.bool),1)
    scores = scores.masked_fill(mask, float("-inf"))
    A = torch.softmax(scores, -1)
    return torch.einsum("bhts,bshd->bthd", A, V)

def _self_check():
    Q = torch.randn(2,6,4,8)
    K = torch.randn(2,6,4,8)
    V = torch.randn(2,6,4,8)
    try:
        out = mh_causal_attn(Q,K,V)
    except NotImplementedError:
        print("🔧  Implement mh_causal_attn.")
        return
    assert torch.allclose(out, _ref_mh_attn(Q,K,V), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 25 – Shannon entropy
----------------------------
Given a batch of **probability distributions** `p` whose entries
sum to 1 along a chosen dimension `dim` (default = −1), compute

        H(p) = − Σᵢ pᵢ log pᵢ          (natural log)

Return a tensor with one fewer dimension than the input, the
summation being performed over `dim`.

* Use **no Python for-loops**.
* Add a small ε before log to avoid `log(0)`.

In [3]:
import torch

def entropy(p: torch.Tensor, dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    """Implement me!"""
    assert torch.all(p >= 0) and torch.allclose(p.sum(dim=dim), torch.tensor(1.0)), "invalid input"
    return (-p * torch.log(p + eps)).sum(dim=dim)
    
# ---------- reference + tests ----------
def _ref_entropy(p, dim=-1, eps=1e-8):
    return -(p * (p + eps).log()).sum(dim)

def _self_check():
    p = torch.tensor([[0.5, 0.5],
                      [0.25, 0.75]])
    try:
        H = entropy(p)
    except NotImplementedError:
        print("🔧  Implement entropy and re-run the cell.")
        return
    assert torch.allclose(H, _ref_entropy(p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 26 – Cross-entropy
--------------------------
Given two *matching* probability tensors **q** (the “true” distribution)
and **p** (the “predicted” distribution) of shape `(..., C)`, compute

        CE(q, p) = − Σᵢ qᵢ log pᵢ

along `dim` (default = −1).  Return a tensor of shape equal to
`q` with that dimension removed.

Assume `q` already sums to 1 (it can be soft or one-hot).
No loops; add ε for log-safety.

In [4]:
import torch

def cross_entropy(q: torch.Tensor, p: torch.Tensor,
                  dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    return (-q * torch.log(p + eps)).sum(dim=dim)


def _ref_cross_entropy(q, p, dim=-1, eps=1e-8):
    return -(q * (p + eps).log()).sum(dim)

def _self_check():
    q = torch.tensor([[0., 1., 0.],
                      [0.2, 0.3, 0.5]])
    p = torch.tensor([[0.1, 0.8, 0.1],
                      [0.3, 0.2, 0.5]])
    try:
        ce = cross_entropy(q, p)
    except NotImplementedError:
        print("🔧  Implement cross_entropy.")
        return
    assert torch.allclose(ce, _ref_cross_entropy(q, p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 27 – KL divergence  D_KL(q || p)
----------------------------------------
For two probability tensors **q** (reference) and **p** (model)
of identical shape `(..., C)`, compute

        KL(q‖p) = Σᵢ qᵢ log(qᵢ / pᵢ)

over `dim` (default = −1).  The output’s shape lacks that dimension.

Requirements
------------
* Pure tensor ops, no loops.
* Add ε to both q and p inside the log to avoid division by zero.

In [5]:
import torch

def kl_divergence(q: torch.Tensor, p: torch.Tensor,
                  dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    return (q * (torch.log(q + eps) - torch.log(p + eps))).sum(dim=dim)


def _ref_kl(q, p, dim=-1, eps=1e-8):
    q_safe = q + eps
    p_safe = p + eps
    return (q_safe * (q_safe / p_safe).log()).sum(dim)

def _self_check():
    q = torch.tensor([[0.6, 0.4]])
    p = torch.tensor([[0.5, 0.5]])
    try:
        kl = kl_divergence(q, p)
    except NotImplementedError:
        print("🔧  Implement kl_divergence.")
        return
    assert torch.allclose(kl, _ref_kl(q, p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 28 – Sinusoidal positional encoding
-------------------------------------------
Implement the original Transformer positional embedding

    PE[pos, 2i]     =  sin( pos / 10000^(2i/d) )
    PE[pos, 2i + 1] =  cos( pos / 10000^(2i/d) )

for sequence length `seq_len` and hidden size `d` (d must be even).

Return a tensor of shape **(seq_len, d)** on the requested
`device` / `dtype` – and **use no Python loops**.

In [7]:
import torch
import math
from typing import Optional

def compute_sin_cos(T, D):
    # sin(t * 10000^(-i/(D/2)))
    # cos(t * 10000^(-i/(D/2)))

    H = D//2
    tt = torch.arange(0, T).view(T, 1)
    ii = torch.arange(0, H).view(1, H)
    ff = 10000**(-ii/H)

    sin = torch.sin(tt * ff)
    cos = torch.cos(tt * ff)

    return sin, cos
    
def sinusoidal_positional_encoding(seq_len: int,
                                   d: int,
                                   device: Optional[torch.device] = None,
                                   dtype: torch.dtype = torch.float32
                                   ) -> torch.Tensor:
    """Implement me!"""
    assert d % 2 == 0, "d must be even"
    sin, cos = compute_sin_cos(seq_len, d)
    pe = torch.stack((sin, cos), dim=-1).reshape(seq_len, d)
    return pe


# -----------------------------------------------------------
def _ref_sinusoidal(seq_len, d, device=None, dtype=torch.float32):
    if d % 2:
        raise ValueError("d must be even.")
    pos = torch.arange(seq_len, device=device).unsqueeze(1)          # (T,1)
    i   = torch.arange(0, d, 2, device=device).unsqueeze(0)          # (1,d/2)
    angle = pos / torch.pow(10000.0, i / d)                          # (T,d/2)
    pe = torch.zeros(seq_len, d, device=device, dtype=dtype)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe

def _self_check():
    T, D = 5, 8
    try:
        pe = sinusoidal_positional_encoding(T, D)
    except NotImplementedError:
        print("🔧  Implement sinusoidal_positional_encoding.")
        return
    assert pe.shape == (T, D), "Shape mismatch"
    assert torch.allclose(pe, _ref_sinusoidal(T, D), atol=1e-6), "Wrong values"
    # Sanity: first position should be [0,1,0,1,...]
    first = pe[0]
    assert torch.allclose(first[0::2], torch.zeros(D//2))            # sin(0)=0
    assert torch.allclose(first[1::2], torch.ones (D//2))            # cos(0)=1
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 29 – Rotary positional embedding (RoPE)
-----------------------------------------------
Apply rotary position encodings (Su et al., 2021) to **query** and
**key** tensors of shape (B, T, d) where d is even.

Algorithm (vector form)
~~~~~~~~~~~~~~~~~~~~~~~
Split last dim into even/odd pairs:

    x_even, x_odd = x[..., 0::2], x[..., 1::2]

Define frequency vector:
    θ_i = 10000^{ -2i / d }   for i = 0 … d/2−1

For each position **pos** (0-based):

    sin  = sin(pos * θ)
    cos  = cos(pos * θ)

Rotate each pair:

    x_rot_even =  x_even * cos - x_odd * sin
    x_rot_odd  =  x_even * sin + x_odd * cos

Interleave the two parts back together to shape (B,T,d).

Return **(q_rot, k_rot)**.

Requirements
------------
* Pure tensor ops, **no explicit Python loops**.

In [10]:
import torch
import math
from typing import Tuple

def compute_sin_cos(T, D):
    # sin(t * 10000^(-i/(D/2)))
    # cos(t * 10000^(-i/(D/2)))

    H = D//2
    tt = torch.arange(0, T).view(T, 1)
    ii = torch.arange(0, H).view(1, H)
    ff = 10000**(-ii/H)

    sin = torch.sin(tt * ff)
    cos = torch.cos(tt * ff)

    return sin, cos

def apply_rope(q: torch.Tensor,
               k: torch.Tensor
               ) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Args
    ----
    q, k : (B, T, d)  with even d

    Returns
    -------
    q_rot, k_rot  – same shapes after rotary encoding.
    """
    B, T, D = q.shape
    sin, cos = compute_sin_cos(T, D)

    def rope(x):
        x0 = x[:, :, 0::2]
        x1 = x[:, :, 1::2]

        y0 = x0 * cos - x1 * sin
        y1 = x0 * sin + x1 * cos

        y = torch.stack((y0, y1), dim=-1).reshape(B, T, D)

        return y

    return rope(q), rope(k)

# -----------------------------------------------------------
def _rope_rotate(x):
    B,T,d = x.shape
    half  = d // 2
    freqs = torch.pow(10000.0, -torch.arange(0, half, device=x.device) / half)
    pos   = torch.arange(T, device=x.device).float()                # (T,)
    sin   = torch.sin( pos.unsqueeze(1) * freqs )                   # (T,half)
    cos   = torch.cos( pos.unsqueeze(1) * freqs )
    sin, cos = sin.unsqueeze(0), cos.unsqueeze(0)                   # (1,T,half)
    x_even, x_odd = x[..., 0::2], x[..., 1::2]                      # (B,T,half)
    rot_even = x_even * cos - x_odd * sin
    rot_odd  = x_even * sin + x_odd * cos
    # interleave
    x_rot = torch.empty_like(x)
    x_rot[..., 0::2] = rot_even
    x_rot[..., 1::2] = rot_odd
    return x_rot

def _ref_apply_rope(q, k):
    return _rope_rotate(q), _rope_rotate(k)

def _self_check():
    q = torch.randn(2, 7, 12)
    k = torch.randn(2, 7, 12)
    try:
        q_r, k_r = apply_rope(q, k)
    except NotImplementedError:
        print("🔧  Implement apply_rope.")
        return
    q_ref, k_ref = _ref_apply_rope(q, k)
    assert torch.allclose(q_r, q_ref, atol=1e-6)
    assert torch.allclose(k_r, k_ref, atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 30 – Connectionist-Temporal Classification (CTC) Loss
------------------------------------------------------------
Write CTC loss **from scratch** (forward + backward) using PyTorch’s
low-level `torch.autograd.Function`.

•  **Signature** must match `torch.nn.CTCLoss`:
       loss = CustomCTCLoss(blank=0, reduction="mean", zero_infinity=False)
       loss(log_probs, targets, input_lengths, target_lengths)

•  **log_probs**  (T, N, C) — log-softmax probabilities  
   **targets**    (N, S)   — padded with a special value (default 0)  
   **input_lengths**  (N,) — actual lengths ≤ T  
   **target_lengths** (N,) — ≤ S

•  Implement the *exclusive* β-variable version of the backward dynamic
   program (β_t(s) excludes the current label at time t).

•  No need to handle GPU / AMP edge cases; loops are fine for the small
  unit tests below, but try to keep your code vectorised where you can.

-----------------------------------------------------------------
Skeleton
~~~~~~~~
1.  **forward**
      – compute CTC loss for each batch element  
      – save anything you need on ctx for backward  
      – return either mean or sum according to `reduction`

2.  **backward**
      – input grad is `grad_out * dLoss/dLogProbs` (shape = log_probs)  
      – return gradients in the order of forward’s inputs  
      – all grads for integer tensors must be **None**

Tips
~~~~
* Expand each target with blanks: e.g. target=[g,a,t] → [blk,g,blk,a,blk,t,blk].
* α/β recurrences (log-space) are stable:
      α_t(s) = logsumexp( α_{t-1}(s)        + lp_t(s),
                           α_{t-1}(s-1)      + lp_t(s),
                           α_{t-1}(s-2) (if label changes) + lp_t(s) )
      β_t(s) = logsumexp( β_{t+1}(s)        + lp_{t+1}(s),
                           β_{t+1}(s+1)      + lp_{t+1}(s+1),
                           β_{t+1}(s+2) (if label changes) + lp_{t+1}(s+2) )
  with **exclusive** β meaning β_t does *not* include lp_t.
* The gradient wrt log_probs is  exp(α+β-logZ), reduced over s.

Feel free to peek at `torch.nn.functional.ctc_loss` for reference, but
do **not** call it inside your implementation.

------------------------------------------------------------
Starter code below — fill in the two `raise NotImplementedError` blocks.

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

NUM_INF = torch.tensor(1e20)

def extend_target(target, blank=0):
    L = len(target)
    out = torch.full((2*L+1,), blank)
    out[1::2] = target
    return out

def logsumexp(a, b):
    a, b = torch.maximum(a, b), torch.minimum(a, b)
    return a + torch.log1p(torch.exp(b-a))

# -------------------  YOUR IMPLEMENTATION AREA  -------------------
class CustomCTCLossFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                log_probs: torch.Tensor,        # (maxT,N,C)  log-softmax already
                targets: torch.Tensor,          # (N,maxL)
                input_lengths: torch.Tensor,    # (N,)
                target_lengths: torch.Tensor,   # (N,)
                blank: int,
                reduction: str,
                zero_infinity: bool,
                gradient_version: str) -> torch.Tensor:
        """
        Returns a scalar (if reduction != 'none') or shape (N,) tensor.
        Save everything you need on ctx for backward.
        """
        maxT, N, C = log_probs.shape
        N, maxL = targets.shape

        losses = []
        grads = torch.zeros_like(log_probs)
        for n in range(N):
            T = input_lengths[n]
            L = target_lengths[n]
            S = 2*L + 1

            lgp = log_probs[:T, n, :]             # T, C
            tgt = targets[n, :L]                  # L
            ext = extend_target(tgt, blank=blank) # 2L+1

            # compute alpha
            alpha = torch.full((T, S), -NUM_INF, dtype=log_probs.dtype)
            alpha[0, 0] = lgp[0, ext[0]]
            alpha[0, 1] = lgp[0, ext[1]]
            #
            for t in range(1, T):
                for s in range(S):
                    # frame t, emission ext[s]
                    stay = alpha[t-1, s-0] + lgp[t, ext[s]]
                    add1 = alpha[t-1, s-1] + lgp[t, ext[s]] if (s-1 >= 0) else -NUM_INF
                    add2 = alpha[t-1, s-2] + lgp[t, ext[s]] if (s-2 >= 0 and ext[s-1] == blank and ext[s-2] != ext[s]) else -NUM_INF

                    alpha[t, s] = logsumexp(logsumexp(stay, add1), add2)

            logZa = logsumexp(alpha[T-1, S-2], alpha[T-1, S-1])

            # compute beta
            beta = torch.full((T, S), -NUM_INF, dtype=log_probs.dtype)
            beta[T-1, S-2] = 0
            beta[T-1, S-1] = 0
            #
            for t in reversed(range(T-1)):
                for s in reversed(range(S)):
                    # frame t, multiple emission possibility
                    stay = lgp[t+1, ext[s+0]] + beta[t+1, s+0]
                    add1 = lgp[t+1, ext[s+1]] + beta[t+1, s+1] if (s+1 < S) else -NUM_INF
                    add2 = lgp[t+1, ext[s+2]] + beta[t+1, s+2] if (s+2 < S and ext[s+1] == blank and ext[s+2] != ext[s]) else -NUM_INF

                    beta[t, s] = logsumexp(logsumexp(stay, add1), add2)

            logZb = logsumexp(lgp[0, ext[0]] + beta[0, 0], lgp[0, ext[1]] + beta[0, 1])

            # compute loss
            assert torch.allclose(logZa, logZb), f"mismatch from alpha and beta computation, {logZa}, {logZb}"
            logZ = (logZa + logZb) / 2
            losses.append(-logZ)

            # compute gamma
            gamma = alpha + beta - logZ

            for s in range(S):
                grads[:T, n, ext[s]] -= torch.exp(gamma[:, s])

            if gradient_version == "torch":
                grads[:T, n, :] += torch.exp(lgp[:T, :])

        losses = torch.tensor(losses)

        ctx.save_for_backward(grads)

        return losses

    
    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        grad_out is either shape (N,) or scalar depending on reduction.
        Must return gradients for the *same* number of inputs received
        in forward:  log_probs, targets, input_lengths, target_lengths,
        blank, reduction, zero_infinity.  Gradients for non-tensor or
        int/str args should be None.
        """

        N, = grad_out.shape
        grads, = ctx.saved_tensors
        grad_in = grad_out.reshape(1, N, 1) * grads
        
        return grad_in, None, None, None, None, None, None, None


class CustomCTCLoss(nn.Module):
    """
    Thin wrapper to match nn.CTCLoss exact call signature.
    """
    def __init__(self,
                 blank: int = 0,
                 reduction: str = "mean",
                 zero_infinity: bool = False,
                 gradient_version: str = "true"):
        super().__init__()
        if reduction not in ("none", "mean", "sum"):
            raise ValueError("reduction must be 'none', 'mean' or 'sum'")
        if gradient_version not in ("true", "torch"):
            raise ValueError("gradient_version must be 'true' or 'torch'")
        self.blank = blank
        self.reduction = reduction
        self.zero_infinity = zero_infinity
        self.gradient_version = gradient_version

    def forward(self,
                log_probs: torch.Tensor,
                targets: torch.Tensor,
                input_lengths: torch.Tensor,
                target_lengths: torch.Tensor) -> torch.Tensor:
        losses = CustomCTCLossFunction.apply(
            log_probs, targets, input_lengths, target_lengths,
            self.blank, self.reduction, self.zero_infinity, self.gradient_version
        )
        if self.reduction == "none":
            loss = losses
        elif self.reduction == "mean":
            loss = torch.mean(losses / target_lengths)
        elif self.reduction == "sum":
            loss = torch.sum(losses)
        return loss
# ------------------------------------------------------------------


# -------------------  REFERENCE + SELF-CHECK  ---------------------
def _ref_ctc_loss(log_probs, targets, input_lengths, target_lengths,
                  blank=0, reduction="mean", zero_infinity=False):
    return F.ctc_loss(log_probs, targets, input_lengths, target_lengths,
                      blank=blank, reduction=reduction,
                      zero_infinity=zero_infinity)

def _self_check():
    torch.manual_seed(0)
    T, N, C = 20, 3, 4
    S = 9
    log_probs = F.log_softmax(torch.randn(T, N, C), dim=-1).requires_grad_()
    targets = torch.randint(1, C, (N, S), dtype=torch.long)
    input_lengths  = torch.full((N,), T, dtype=torch.long)
    target_lengths = torch.randint(1, S+1, (N,), dtype=torch.long)

    my_ctc  = CustomCTCLoss(blank=0, reduction="mean", gradient_version="torch")
    ref_ctc = nn.CTCLoss(blank=0, reduction="mean")

    try:
        loss_my  = my_ctc(log_probs, targets, input_lengths, target_lengths)
    except NotImplementedError:
        print("🔧  Implement CustomCTCLossFunction.forward/backward and re-run.")
        return

    loss_ref = ref_ctc(log_probs, targets, input_lengths, target_lengths)

    # Forward value check
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-4), \
        f"Loss mismatch: {loss_my} vs {loss_ref}"

    # Gradient check (wrt log_probs only)
    loss_ref.backward()
    grad_ref = log_probs.grad.clone()
    log_probs.grad.zero_()
    loss_my.backward()
    grad_my = log_probs.grad

    assert torch.allclose(grad_my, grad_ref, atol=1e-4), \
        "Gradient mismatch w.r.t. log_probs"

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


### Note PyTorch implementation of the CTC gradient is incorrect.

In [45]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
T, N, C = 20, 3, 4
S = 9

log_probs = F.log_softmax(torch.randn(T, N, C, dtype=torch.float64), dim=2).requires_grad_()
targets = torch.randint(1, C, (N, S), dtype=torch.long)
input_lengths = torch.full((N,), T, dtype=torch.long)
target_lengths = torch.randint(1, S + 1, (N,), dtype=torch.long)

custom_ctc_loss = CustomCTCLoss(blank=0, reduction="mean", gradient_version="true")

torch.autograd.gradcheck(custom_ctc_loss, (logits, targets, input_lengths, target_lengths), eps=1e-6, atol=1e-4)

True

Problem 31 – InfoNCE loss used in CLIP
-------------------------------------
Given L2-normalised **image** and **text** feature matrices

    images : (B, D)
    texts  : (B, D)

compute the symmetric InfoNCE loss

        L = ½ · [ CE( logits_img2txt, targets ) +
                  CE( logits_txt2img, targets ) ]

where

    logits_img2txt =  (images · textsᵀ) / τ      # (B, B)
    logits_txt2img =  logits_img2txtᵀ
    targets = torch.arange(B)                    # 0 … B−1

`τ` is a positive **temperature** scalar (default 0.07).

Return a scalar loss.  Use **no explicit Python loops**.

**Your task:**
Implement the function `info_nce_clip` below.

Afterwards, the self-check harness will compare your value and
gradient against a reference implementation.
"""

In [6]:
import torch
import torch.nn.functional as F
from typing import Optional

# ------------------------ YOUR CODE HERE ------------------------
def info_nce_clip(image_feats: torch.Tensor,
                  text_feats: torch.Tensor,
                  temperature: float = 0.07) -> torch.Tensor:
    """
    Args
    ----
    image_feats : (B, D)  *L2-normalised*
    text_feats  : (B, D)  *L2-normalised*
    temperature : positive scalar τ

    Returns
    -------
    Scalar InfoNCE loss (float tensor, requires_grad=True if inputs do).
    """
    # image_feats: B, D
    # text_feats: C, D
    B, D = image_feats.shape
    logits_i2t = torch.einsum("bd,cd->bc", image_feats, text_feats) / temperature
    logits_t2i = logits_i2t.permute(1, 0)
    
    targets = torch.arange(B)
    loss_i = F.cross_entropy(logits_i2t, targets, reduction="mean")
    loss_t = F.cross_entropy(logits_t2i, targets, reduction="mean")
    return 0.5 * (loss_i + loss_t)
    
    
    
# -----------------------------------------------------------------


# --------------------- REFERENCE + SELF-CHECK --------------------
def _ref_info_nce_clip(img, txt, tau=0.07):
    B = img.size(0)
    logits = (img @ txt.t()) / tau                # (B,B)
    targets = torch.arange(B, device=img.device)
    loss_i = F.cross_entropy(logits, targets, reduction="mean")
    loss_t = F.cross_entropy(logits.t(), targets, reduction="mean")
    return 0.5 * (loss_i + loss_t)

def _self_check():
    torch.manual_seed(0)
    B, D = 8, 64
    img = F.normalize(torch.randn(B, D), dim=-1, p=2).requires_grad_()
    txt = F.normalize(torch.randn(B, D), dim=-1, p=2)

    try:
        my_loss = info_nce_clip(img, txt)
    except NotImplementedError:
        print("🔧  Implement info_nce_clip and re-run the cell.")
        return

    ref_loss = _ref_info_nce_clip(img, txt)

    # Forward check
    assert torch.allclose(my_loss.detach(), ref_loss, atol=1e-6), \
        f"Loss mismatch: {my_loss} vs {ref_loss}"

    # Gradient check wrt image features
    ref_loss.backward()
    grad_ref = img.grad.clone()
    img.grad.zero_()
    my_loss.backward()
    grad_my = img.grad

    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch w.r.t. image features"

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


Problem 32 – Vectorized K-Means
==============================
Implement the classic **K-Means** clustering algorithm in PyTorch
without any Python loops over the data points or cluster indices.

Input
-----
* **data**  : `(N, D)` float32 tensor (no grad needed)
* **k**     : int  (2 ≤ k ≤ N)
* **num_iters** : int, number of full E→M iterations (default = 20)

Output
------
A tuple `(centroids, labels)` where

* **centroids** : `(k, D)` tensor, sorted by the first dimension
* **labels**    : `(N,)` long tensor giving the cluster index (0-based)
                  that each point belongs to.

Algorithm
---------
1. **Initialisation** – take the **first k** rows of `data` as the
   starting centroids (deterministic for unit tests).
2. Loop `num_iters` times  
   ‣ **E-step** : assign each point to its nearest centroid  
   ‣ **M-step** : recompute each centroid as the mean of its assigned
     points (guaranteed non-empty with the chosen initialisation).
3. Return final centroids (sorted by row index) and labels.

Implementation rules
--------------------
* You MAY loop over *iterations* (`for _ in range(num_iters):`)
  but **MUST NOT** loop over N or k.
* All heavy math must be done with tensor ops/broadcasting.

In [42]:
import torch
from typing import Tuple

# -------------------------- YOUR CODE ---------------------------
def kmeans(
    data: torch.Tensor,
    k: int,
    num_iters: int = 20
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Vectorised K-Means.

    Returns
    -------
    centroids : (k, D)  tensor
    labels    : (N,)    long tensor
    """
    
    N, D = data.shape
    centroids = data[:k].clone()
    
    for _ in range(num_iters):
        distance = torch.norm(data.view(N, 1, D) - centroids.view(1, k, D), dim=-1) # N, k
        _, labels = torch.min(distance, dim=-1)
        centroids.scatter_reduce_(dim=0, index=labels.view(N, 1).expand(N, k), src=data, reduce="mean", include_self=False)
        
    centroids, labels = deterministic_sort(centroids, labels)
    return centroids, labels
    
# -----------------------------------------------------------------

def deterministic_sort(centroids, labels):
    # sort centroids to make deterministic comparison easier
    order = centroids[:, 0].argsort()
    remap = order.argsort()
    centroids = centroids[order]
    labels = remap[labels]       # old labels to new labels
    return centroids, labels

# -------------------- Reference & Self-Check ---------------------
def _ref_kmeans(data: torch.Tensor, k: int, num_iters: int = 20):
    N, D = data.shape
    centroids = data[:k].clone()                       # (k,D)

    for _ in range(num_iters):
        # squared distances: (N,k)
        dists = ((data.unsqueeze(1) - centroids.unsqueeze(0)) ** 2).sum(-1)
        labels = dists.argmin(dim=1)                  # (N,)

        # recompute centroids
        eye = torch.eye(k, device=data.device)[labels]        # (N,k) one-hot
        counts = eye.sum(0)                                  # (k,)
        sums   = eye.T @ data                                # (k,D)
        centroids = sums / counts.unsqueeze(1)

    centroids, labels = deterministic_sort(centroids, labels)
    return centroids, labels


def _self_check():
    torch.manual_seed(0)
    # synthetic 2-cluster toy data
    N, D, k = 50, 2, 2
    data = torch.cat([torch.randn(N // 2, D) * 0.2 + torch.tensor([2, 0]),
                      torch.randn(N // 2, D) * 0.3 + torch.tensor([-2, 0])])

    try:
        cent_my, lbl_my = kmeans(data, k, num_iters=15)
    except NotImplementedError:
        print("🔧  Implement kmeans and re-run the cell.")
        return

    cent_ref, lbl_ref = _ref_kmeans(data, k, num_iters=15)

    # Compare centroids (order-sorted)
    assert torch.allclose(cent_my, cent_ref, atol=1e-3), \
        "Centroids differ from reference."

    # Compare cluster assignment consistency
    assert (lbl_my == lbl_ref).all(), "Labels differ from reference."

    print("✅  basic K-Means forward test passed")

_self_check()

✅  basic K-Means forward test passed


Problem 33 – Linear regression (no bias)
=======================================
Implement a *custom autograd* `Function` that returns the mean-squared
error (MSE) loss for a linear model **without an intercept**:

    ŷ = X @ w                 # X: (N, D), w: (D,)
    L  = ½ · mean( (ŷ − y)² )   (scalar)

Your task
---------
1. Fill in `LinearMSEFunction.forward` — compute the scalar loss.
2. Fill in `LinearMSEFunction.backward` — compute ∂L/∂w only
   (return `None` for the non-learnable inputs X and y).

Vectorised gradient formula
---------------------------
With the ½ scaling, the gradient is

    ∂L/∂w = (1/N) · Xᵀ · (ŷ − y)        # shape (D,)

No explicit loops over N or D.

In [48]:
import torch
from typing import Optional, Tuple

import torch
from typing import Tuple, Optional

# ----------------------------- YOUR CODE ------------------------
class LinearMSEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                w: torch.Tensor,    # (D,)  requires_grad
                X: torch.Tensor,    # (N,D)
                y: torch.Tensor):   # (N,)
        """
        Return scalar loss.  Save tensors needed for backward in ctx.
        """
        y_pred = X @ w
        loss = 0.5 * (y_pred - y).pow(2).mean()
        
        ctx.save_for_backward(w, X, y)
        
        return loss
        

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        grad_out is a scalar (shape []); propagate it to ∂L/∂w.
        Return grads w.r.t. (w, X, y) in that order.
        """
        w, X, y = ctx.saved_tensors
        
        N, D = X.shape
        
        return (X.T @ X @ w - X.T @ y) / N, None, None


def linear_mse_loss(w: torch.Tensor,
                    X: torch.Tensor,
                    y: torch.Tensor) -> torch.Tensor:
    """Convenience wrapper."""
    return LinearMSEFunction.apply(w, X, y)
# ----------------------------------------------------------------


# --------------------- Reference + Self-Check -------------------
def _ref_linear_mse(w, X, y):
    y_hat = X @ w
    return 0.5 * (y_hat - y).pow(2).mean()

def _self_check():
    torch.manual_seed(0)
    N, D = 13, 5
    X = torch.randn(N, D)
    y = torch.randn(N)
    w = torch.randn(D, requires_grad=True)

    try:
        loss_my = linear_mse_loss(w, X, y)
    except NotImplementedError:
        print("🔧  Fill in LinearMSEFunction forward/backward.")
        return

    loss_ref = _ref_linear_mse(w, X, y)

    # Forward value
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-6), \
        "Loss mismatch."

    # Gradient
    loss_ref.backward()
    grad_ref = w.grad.clone()
    w.grad.zero_()
    loss_my.backward()
    grad_my = w.grad
    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch."

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


Problem 34 – Binary logistic regression (no bias)
================================================
Implement a custom autograd `Function` that computes the *binary
cross-entropy* (BCE) loss for a sigmoid model without bias:

    p   = σ( X @ w )            # σ is sigmoid
    L   = mean(  − y·log p  − (1−y)·log(1−p) )    (scalar)

where y ∈ {0,1}.  No explicit loops over N or D.

Gradient formula
----------------
∂L/∂w = (1/N) · Xᵀ · (p − y)      # shape (D,)

In [54]:
import torch
import torch.nn.functional as F
from typing import Tuple, Optional

# ----------------------------- YOUR CODE ------------------------
class LogisticBCEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                w: torch.Tensor,   # (D,)  requires_grad
                X: torch.Tensor,   # (N,D)
                y: torch.Tensor):  # (N,)   0/1 floats
        """
        Save tensors needed for backward. Return scalar BCE loss.
        """
        p = torch.sigmoid(X @ w)
        loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()
        ctx.save_for_backward(X, y, p)
        return loss
        

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        Return grads for (w, X, y) in order.  Only w gets a gradient.
        """
        X, y, p = ctx.saved_tensors
        grad = -((y - p).view(-1, 1) * X).mean(dim=0) # N
        return grad, None, None


def logistic_bce_loss(w: torch.Tensor,
                      X: torch.Tensor,
                      y: torch.Tensor) -> torch.Tensor:
    return LogisticBCEFunction.apply(w, X, y)
# ----------------------------------------------------------------


# --------------------- Reference + Self-Check -------------------
def _ref_logistic_bce(w, X, y):
    p = torch.sigmoid(X @ w)
    return F.binary_cross_entropy(p, y, reduction="mean")

def _self_check():
    torch.manual_seed(1)
    N, D = 17, 4
    X = torch.randn(N, D)
    y = torch.randint(0, 2, (N,)).float()
    w = torch.randn(D, requires_grad=True)

    try:
        loss_my = logistic_bce_loss(w, X, y)
    except NotImplementedError:
        print("🔧  Fill in LogisticBCEFunction forward/backward.")
        return

    loss_ref = _ref_logistic_bce(w, X, y)

    # Forward check
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-6), \
        "Loss mismatch."

    # Gradient check
    loss_ref.backward()
    grad_ref = w.grad.clone()
    w.grad.zero_()
    loss_my.backward()
    grad_my = w.grad
    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch."

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


Problem 35 – Soft-max Regression Loss with Manual Gradient
=========================================================
Write a custom autograd.Function that implements the mean cross-entropy
loss for *bias-free* multiclass logistic regression.

Definitions
-----------
Inputs
  • **W**  – weights, shape **(D, C)**, `requires_grad=True`
  • **X**  – features, shape **(N, D)**
  • **y**  – integer class labels, shape **(N,)**, values ∈ {0,…,C−1}

Logits
       Z = X · W                         # (N, C)

Soft-max probabilities
       P = softmax(Z, dim=1)             # (N, C)

Loss  (mean negative log-likelihood)
       L =  (1/N) · Σ_i  −log P[i, y_i]

Gradient w.r.t. W
-----------------
Let **Y_onehot** be the one-hot matrix of `y` (N×C).  
Then the analytical gradient is

       ∂L/∂W = (1/N) · Xᵀ · (P − Y_onehot)      # shape (D, C)

Your Task
~~~~~~~~~
Implement **both** `forward` and `backward` in `SoftmaxRegLossFn`
using the formula above.  
Gradients for `X` and `y` are **None** (they’re treated as constants).

In [2]:
import torch
import torch.nn.functional as F
from typing import Tuple, Optional

# ------------------------ YOUR CODE ------------------------
class SoftmaxRegLossFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                W: torch.Tensor,          # (D, C)
                X: torch.Tensor,          # (N, D)
                y: torch.Tensor           # (N,)
                ) -> torch.Tensor:
        
        D, C = W.shape
        N, D = X.shape
        
        Z = X @ W
        P = torch.softmax(Z, dim=-1)
        I = torch.eye(C, C)
        
        Y = I[y]
        
        L = -(Y * torch.log(P)).sum() / N
        
        ctx.save_for_backward(X, Y, P)
        return L

    @staticmethod
    def backward(ctx,
                 grad_out: torch.Tensor
                 ) -> Tuple[Optional[torch.Tensor], None, None]:
        
        X, Y, P = ctx.saved_tensors

        N, D = X.shape
        
        grad = X.T @ (P - Y) / N
        
        return grad, None, None


def softmax_reg_loss(W: torch.Tensor,
                     X: torch.Tensor,
                     y: torch.Tensor) -> torch.Tensor:
    """Convenience wrapper around the custom autograd Function."""
    return SoftmaxRegLossFn.apply(W, X, y)
# -----------------------------------------------------------------


# --------------- Reference implementation & self-check -----------
def _ref_softmax_loss(W, X, y):
    logits = X @ W
    return F.cross_entropy(logits, y, reduction="mean")

def _self_check_softmax():
    torch.manual_seed(0)
    N, D, C = 40, 6, 5
    X = torch.randn(N, D)
    W = torch.randn(D, C, requires_grad=True)
    y = torch.randint(0, C, (N,))

    try:
        my_loss = softmax_reg_loss(W, X, y)
    except NotImplementedError:
        print("🔧  Implement SoftmaxRegLossFn forward/backward.")
        return

    ref_loss = _ref_softmax_loss(W, X, y)

    # Forward check
    assert torch.allclose(my_loss.detach(), ref_loss, atol=1e-6), \
        f"Loss mismatch: {my_loss} vs {ref_loss}"

    # Gradient check w.r.t. W
    ref_loss.backward()
    grad_ref = W.grad.clone()
    W.grad.zero_()
    my_loss.backward()
    grad_my = W.grad
    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch on W"

    print("✅  Multiclass logistic regression forward & backward OK")

_self_check_softmax()

✅  Multiclass logistic regression forward & backward OK


Problem 36 – Adam Optimizer: Derivation + Implementation
=======================================================
**Goal** – Write your own Adam optimizer and show it gives the same
parameter trajectory as `torch.optim.Adam`.

Background & Update Rule
------------------------
For each parameter **w** and its gradient **gₜ = ∇ₜ L(w)** at step *t*
(starting at t=1):

1.  First-order moment (exponential moving average)
        mₜ  = β₁ · mₜ₋₁  +  (1 − β₁) · gₜ

2.  Second-order moment (uncentered variance)
        vₜ  = β₂ · vₜ₋₁  +  (1 − β₂) · gₜ²      (element-wise square)

3.  ***Bias correction***
        m̂ₜ = mₜ / (1 − β₁ᵗ)
        v̂ₜ = vₜ / (1 − β₂ᵗ)

4.  Parameter update (no bias term, no weight decay)
        w  ←  w  −  lr · m̂ₜ / ( √( v̂ₜ ) + ε )

Your task
~~~~~~~~~
Implement the `step` method of `CustomAdam` below so the rule above is
applied to **every** tensor in `self.params` each call.

Implementation notes
--------------------
* You MAY use simple Python loops over the parameter list.
* Do **not** add weight-decay, AMSGrad, or per-parameter learning rates;
  just the plain Adam defined above.
* The optimizer keeps **state dicts**  `{"step": t, "m": …, "v": …}`
  per parameter (similar to PyTorch).

Self-check
~~~~~~~~~~
The harness compares your parameter values to PyTorch’s Adam after
20 steps on a tiny regression toy problem.

Fill in the `raise NotImplementedError` and re-run the cell.

In [7]:
import torch
import torch.nn.functional as F
from typing import List, Dict, Any

# -------------------------- YOUR CODE -----------------------------
class CustomAdam:
    def __init__(self,
                 params: List[torch.Tensor],
                 lr: float = 1e-3,
                 betas: tuple = (0.9, 0.999),
                 eps: float = 1e-8):
        self.params = list(params)
        self.lr     = lr
        self.beta1, self.beta2 = betas
        self.eps    = eps
        # one state dict per tensor
        self.state: List[Dict[str, Any]] = [
            {"step": 0,
             "m": torch.zeros_like(p),
             "v": torch.zeros_like(p)}
            for p in self.params
        ]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    @torch.no_grad()
    def step(self):
        """
        Update every parameter **in-place** using the Adam rule above.
        All math must happen with tensor operations.
        """
        for p, dic in zip(self.params, self.state):
            m = self.beta1 * dic["m"] + (1 - self.beta1) * p.grad
            v = self.beta2 * dic["v"] + (1 - self.beta2) * p.grad.pow(2)
            mh = m / (1 - self.beta1**(dic["step"]+1))
            vh = v / (1 - self.beta2**(dic["step"]+1))
            p.add_(-self.lr * mh / (vh + self.eps).pow(0.5))
            dic["step"] += 1
            dic["m"] = m
            dic["v"] = v
        
        
        
# ------------------------------------------------------------------


# ----------------------  SELF-CHECK HARNESS -----------------------
def _self_check_adam():
    torch.manual_seed(0)
    N, D = 32, 4
    X = torch.randn(N, D)
    true_w = torch.randn(D)
    y = X @ true_w + 0.1 * torch.randn(N)

    # model parameter (single tensor)
    w_ref = torch.randn(D, requires_grad=True)
    w_my  = w_ref.clone().detach().requires_grad_()

    # PyTorch Adam
    opt_ref = torch.optim.Adam([w_ref], lr=1e-2, betas=(0.9, 0.999),
                               eps=1e-8, weight_decay=0.0)

    # Your Adam
    opt_my  = CustomAdam([w_my], lr=1e-2, betas=(0.9, 0.999), eps=1e-8)

    try:
        # 20 optimisation steps
        for _ in range(20):
            # --- reference optimiser ---
            opt_ref.zero_grad()
            loss_ref = 0.5 * ((X @ w_ref - y) ** 2).mean()
            loss_ref.backward()
            opt_ref.step()

            # --- your optimiser ---
            opt_my.zero_grad()
            loss_my = 0.5 * ((X @ w_my - y) ** 2).mean()
            loss_my.backward()
            opt_my.step()
    except NotImplementedError:
        print("🔧  Implement CustomAdam.step() and re-run.")
        return

    # Compare parameters
    if torch.allclose(w_my, w_ref, atol=1e-6):
        print("✅  Custom Adam matches PyTorch Adam after 20 steps")
    else:
        max_diff = (w_my - w_ref).abs().max()
        print(f"❌  Parameter mismatch. Max abs diff = {max_diff:.2e}")

_self_check_adam()

✅  Custom Adam matches PyTorch Adam after 20 steps


Problem 37 – Top-k logits filter
================================
In many decoder-only LLMs, sampling first **keeps only the k highest
log-probability tokens** per batch element and sets every other logit
to −∞ so their soft-max mass becomes 0.

Task
----
Implement

        filtered = topk_filter(logits, k)

where

    • logits   : (B, V)   float tensor (NOT log-softmaxed)
    • k        : int, 1 ≤ k ≤ V

and

    filtered[b, j] = logits[b, j]   if j is in top-k of row b
                   = −∞             otherwise

No explicit Python loops over B or V.

In [1]:
import torch

def topk_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    # logits: B, V
    B, V = logits.shape
    values, indices = torch.topk(input=logits, k=k, largest=True) # B, k
    out = torch.ones_like(logits) * (-torch.inf)
    out.scatter_(dim=1, index=indices, src=values)
    return out


# ---------------- reference + test ----------------
def _ref_topk_filter(L, k):
    vals, idx = L.topk(k, dim=1)
    out = torch.full_like(L, float("-inf"))
    out.scatter_(1, idx, L.gather(1, idx))
    return out

def _self_check():
    torch.manual_seed(0)
    L = torch.randn(3, 10)
    k = 4
    try:
        out = topk_filter(L, k)
    except NotImplementedError:
        print("🔧  Implement topk_filter.")
        return
    assert torch.allclose(out, _ref_topk_filter(L, k)), "Mismatch"
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 38 – RMSNorm forward
============================
For hidden size *d* the RMSNorm output is

        rms  = √( mean_i  x_i² )          # shape (B, 1)
        y    =  x / rms   *   weight      # (B, d)

Inputs
------
* x      : (B, d)    float tensor
* weight : (d,)      learnable gain (γ)

Return y with the same shape as x.

No loops, use eps=1e-8 for stability.

In [4]:
import torch

def rms_norm(x: torch.Tensor,
             weight: torch.Tensor,
             eps: float = 1e-8) -> torch.Tensor:
    # x: B, d
    B, d = x.shape
    rms = (torch.mean(x.pow(2), dim=1, keepdim=True) + eps).pow(0.5)
    y = x / rms * weight.view(1, d)
    return y
    


# reference & test
def _ref_rms_norm(x, w, eps=1e-8):
    rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + eps)
    return x / rms * w

def _self_check():
    B, d = 5, 7
    x = torch.randn(B, d)
    w = torch.randn(d)
    try:
        y = rms_norm(x, w)
    except NotImplementedError:
        print("🔧  Implement rms_norm.")
        return
    assert torch.allclose(y, _ref_rms_norm(x, w), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 39 – Simulate INT8 matmul
================================
Implement a *toy* symmetric quantisation pipeline:

1.  **quantise(x)**   →  q, scale  
        scale = max(|x|) / 127  
        q     = round(x / scale).clamp(-127, 127).to(torch.int8)

2.  **int8_matmul(A, B)**  
        *Quantise* A and B ⇒ qA, qB, scales sA, sB  
        *Multiply*   int32 out_q = qA @ qB.T         (standard matmul)  
        *Dequantise* out = (sA * sB) * out_q.float()

Return the float32 product that approximates A @ Bᵀ.

No loops over rows/cols.  Ignore zero-points and per-channel scales.

In [15]:
import torch

def quantise_sym(x: torch.Tensor):
    scale_float = torch.max(torch.abs(x), dim=-1, keepdim=True).values / 127
    q_int8 = torch.round(x / scale_float).to(dtype=torch.int8)
    return q_int8, scale_float

def int8_matmul(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    A_quant, A_scale = quantise_sym(A)
    B_quant, B_scale = quantise_sym(B)
    return (A_quant.to(dtype=torch.int32) @ B_quant.to(dtype=torch.int32).T) * (A_scale @ B_scale.T)


# reference & test
def _ref_quant(x):
    s = x.abs().max() / 127
    q = (x / s).round().clamp(-127, 127).to(torch.int8)
    return q, s

def _ref_int8_mm(A, B):
    qA, sA = _ref_quant(A)
    qB, sB = _ref_quant(B)
    out_q  = torch.matmul(qA.float(), qB.float().T)   # int8→fp32
    return (sA * sB) * out_q

def _self_check():
    torch.manual_seed(0)
    A = torch.randn(8, 16)
    B = torch.randn(10, 16)
    try:
        out = int8_matmul(A, B)
    except NotImplementedError:
        print("🔧  Implement quantise_sym and int8_matmul.")
        return
    ref = _ref_int8_mm(A, B)
    # Quantised matmul is an approximation; allow ~1% relative error
    rel_err = (out - ref).abs().mean() / ref.abs().mean()
    assert rel_err < 0.01, f"Rel error too high: {rel_err:.3%}"
    print("✅  basic tests passed (mean rel-err ≈ %.2f%%)" % (rel_err*100))

_self_check()

✅  basic tests passed (mean rel-err ≈ 0.98%)


Problem 40 – Bias-GeLU fusion
============================
Given an input tensor **x** (N, D) and bias **b** (D,), return

        y = GeLU( x + b )

using the exact formulation of `torch.nn.functional.gelu`
(`approximate="none"`).  One kernel call only; **no explicit addition
followed by GeLU** — fuse them in a single expression.

In [17]:
import torch
import torch.nn.functional as F

def bias_gelu(x: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return F.gelu(x + bias)

# reference + test
def _ref_bias_gelu(x, b):
    return F.gelu(x + b)

def _self_check():
    x = torch.randn(4, 6)
    b = torch.randn(6)
    try:
        y = bias_gelu(x, b)
    except NotImplementedError:
        print("🔧  Implement bias_gelu.")
        return
    assert torch.allclose(y, _ref_bias_gelu(x, b), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 41 – Local sliding-window attention
==========================================
Compute **causal, local** self-attention where each token attends only
to the previous **w** tokens (including itself).

Inputs
------
* Q, K, V : (B, T, d)
* w       : window size (int ≥ 1)

Algorithm
---------
For position t, valid keys are K[..., max(0, t−w+1):t+1].
Scores are scaled dot-products (`d**−0.5`) followed by soft-max over
the window, then multiplied with the same V-slice.

Return attention output **O** of shape (B, T, d).

No Python loops over T; vectorise with full (B,T,T) masks
(acceptable for small T in unit tests).

In [22]:
import torch
import math

def sliding_window_attention(Q: torch.Tensor,
                             K: torch.Tensor,
                             V: torch.Tensor,
                             w: int) -> torch.Tensor:
    # Q: B, T, D
    # K: B, S, D
    # V: B, S, D
    B, T, D = Q.shape
    score = torch.einsum("btd,bsd->bts", Q, K) * D**(-0.5) # B, T, S
    t_id = torch.arange(0, T).view(T, 1)
    s_id = torch.arange(0, T).view(1, T).expand(T, T)
    mask = torch.logical_and(torch.le(t_id - w + 1, s_id), torch.le(s_id, t_id))
    score.masked_fill_(~mask, -torch.inf)
    
    attn_weight = torch.softmax(score, dim=-1)
    out = torch.einsum("bts,bsd->btd", attn_weight, V)
    return out
    

# reference + test
def _ref_swa(Q, K, V, w):
    B, T, d = Q.shape
    scale = 1 / math.sqrt(d)
    scores = torch.einsum("btd,bkd->btk", Q, K) * scale   # (B,T,T)
    mask = torch.ones(T, T, device=Q.device, dtype=torch.bool)
    mask = mask.tril().triu(diagonal=-(w-1))  # keep last w
    scores = scores.masked_fill(~mask, float("-inf"))
    attn = torch.softmax(scores, -1)
    return attn @ V

def _self_check():
    torch.manual_seed(0)
    B,T,d,w = 2, 6, 8, 3
    Q = torch.randn(B,T,d)
    K = torch.randn(B,T,d)
    V = torch.randn(B,T,d)
    try:
        out = sliding_window_attention(Q,K,V,w)
    except NotImplementedError:
        print("🔧  Implement sliding_window_attention.")
        return
    assert torch.allclose(out, _ref_swa(Q,K,V,w), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 42 – ALiBi (Attention with Linear Bias)
==============================================
Part A  »  `get_alibi_slopes(num_heads)`  
 Return a 1-D tensor **(H,)** where `s[h] = 2^{-(h / H)}` (the simplified
version used in many open-source models).

Part B  »  `apply_alibi(scores, slopes)`  
 Given raw dot-product scores **(B, H, T, T)** and `slopes (H,)` add

      bias[h, i, j] = - slopes[h] · (T − j − 1)     # distance from j to i,
                                                    # assuming causal i ≥ j

and return the **biased** scores.

No loops over T or H.

In [31]:
import torch

def get_alibi_slopes(num_heads: int) -> torch.Tensor:
    h = torch.arange(num_heads)
    return 2 ** (-h / num_heads)

def apply_alibi(scores: torch.Tensor, slopes: torch.Tensor) -> torch.Tensor:
    B, H, T, _ = scores.shape
    pos = torch.arange(T)
    dist = pos.view(T, 1) - pos.view(1, T)
    bias = -slopes.view(1, H, 1, 1) * dist.view(1, 1, T, T)
    bias = bias.tril()
    return scores + bias

# reference version to compare correctness
def _ref_slopes(H):
    h = torch.arange(H)
    return 2 ** (-h / H)

def _ref_apply(scores, slopes):
    B, H, T, _ = scores.shape
    pos = torch.arange(T, device=scores.device)
    dist = pos.view(1, 1, T, 1) - pos.view(1, 1, 1, T)  # shape (1, 1, T, T)
    bias = -slopes.view(1, H, 1, 1) * dist             # shape (1, H, T, T)
    bias = torch.tril(bias)
    return scores + bias

def _self_check():
    B,H,T = 2,4,5
    s = torch.randn(B,H,T,T)
    try:
        slopes = get_alibi_slopes(H)
        out    = apply_alibi(s, slopes)
    except NotImplementedError:
        print("🔧  Implement get_alibi_slopes / apply_alibi.")
        return
    assert torch.allclose(slopes, _ref_slopes(H), atol=1e-6)
    assert torch.allclose(out, _ref_apply(s, slopes), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 43 – KV-cache append (autoregressive decoding)
=====================================================
Maintain key/value caches of shape (B, H, L, d) where *L* grows with
each generated token.

Task
----
`new_KV = append_kv_cache(cache, new, t)`
* **cache** : (B, H, max_len, d) pre-allocated tensor
* **new**   : (B, H, d) keys OR values for the current step
* **t**     : int index of the *current* position (0-based)

Write a function that **in-place** writes `new` into `cache[..., t, :]`
and returns a (view) slice `(B, H, t+1, d)` representing the valid
portion of the cache.

No Python loops over B,H,d; direct tensor assignment only.
"""

In [1]:
import torch
from typing import Tuple

def append_kv_cache(cache: torch.Tensor,
                    new: torch.Tensor,
                    t: int) -> torch.Tensor:
    B, H, max_len, d = cache.shape
    cache[:, :, t, :] = new
    return cache[:, :, :t+1, :]


# reference + test
def _ref_append(cache, new, t):
    cache[..., t, :].copy_(new)
    return cache[..., :t+1, :]

def _self_check():
    B,H,maxL,d = 2,3,10,4
    K_cache = torch.zeros(B,H,maxL,d)
    newK    = torch.randn(B,H,d)
    t = 0
    try:
        valid = append_kv_cache(K_cache, newK, t)
    except NotImplementedError:
        print("🔧  Implement append_kv_cache.")
        return
    assert torch.allclose(valid, _ref_append(torch.zeros_like(K_cache), newK, t))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 44 – split_heads
=======================
Transform a projection tensor  (B, T, H*d) into (B, H, T, d).

Inputs
------
* x : (B, T, H*d)  contiguous
* H : num heads

Return the reshaped & **transposed** view (avoid `.contiguous()` if
possible).  No loops.

In [2]:
import torch

def split_heads(x: torch.Tensor, H: int) -> torch.Tensor:
    B, T, Hd = x.shape
    x = x.reshape(B, T, H, Hd//H)
    x = x.permute(0, 2, 1, 3)
    return x


# reference + test
def _ref_split(x, H):
    B,T,HD = x.shape
    d = HD // H
    return x.view(B,T,H,d).permute(0,2,1,3)

def _self_check():
    B,T,H,d = 2,5,4,6
    x = torch.randn(B,T,H*d)
    try:
        y = split_heads(x, H)
    except NotImplementedError:
        print("🔧  Implement split_heads.")
        return
    assert torch.allclose(y, _ref_split(x,H))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 45 – combine_heads
==========================
Inverse of `split_heads`: (B, H, T, d) → (B, T, H*d).

Must be the minimal-copy equivalent of `permute` + `reshape`.

In [3]:
import torch

def combine_heads(x: torch.Tensor) -> torch.Tensor:
    B, H, T, d = x.shape
    return x.permute(0, 2, 1, 3).reshape(B, T, H*d)


def _ref_combine(x):
    B,H,T,d = x.shape
    return x.permute(0,2,1,3).reshape(B, T, H*d)

def _self_check():
    B,H,T,d = 2,4,3,5
    x = torch.randn(B,H,T,d)
    try:
        y = combine_heads(x)
    except NotImplementedError:
        print("🔧  Implement combine_heads.")
        return
    assert torch.allclose(y, _ref_combine(x))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 46 – Shared embedding logits
===================================
With weight tying, the token logits are `x @ Wᵀ`.
Implement

        logits = tied_logits(hidden, embed_weight)

where
    • hidden : (B, T, d)
    • embed_weight : (V, d)   (same as used for nn.Embedding)

Return (B, T, V) logits.  Use a single `matmul`; no loops.

In [10]:
import torch

def tied_logits(hidden: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    return hidden @ weight.T

def _ref_tied(h, W):
    return h @ W.t()

def _self_check():
    B,T,d,V = 2,3,8,11
    h = torch.randn(B,T,d)
    W = torch.randn(V,d)
    try:
        out = tied_logits(h,W)
    except NotImplementedError:
        print("🔧  Implement tied_logits.")
        return
    assert torch.allclose(out, _ref_tied(h,W))
    print("✅  basic tests passed")

_self_check()

🔧  Implement tied_logits.


Problem 47 – Fused linear QKV projection
=======================================
Given a single weight **W** of shape (3*d, d_model) and bias
**b** (3*d,), project an input batch **X** (B, T, d_model) into
three tensors Q, K, V each of shape (B, T, d).

Return them as a tuple **(Q, K, V)** with no extra Python loops and
exactly **one** call to `torch.matmul` (or `.linear`).

In [5]:
import torch

def fused_qkv(X: torch.Tensor,
              W: torch.Tensor,
              b: torch.Tensor,
              d: int):
    # X: B, T, D
    # W: 3d, D
    # b: 3d
    
    qkv = X @ W.T + b
    Q, K, V = qkv.chunk(3, dim=-1)
    return Q, K, V
    

# reference + test
def _ref_fused(X,W,b,d):
    proj = X @ W.t() + b
    Q, K, V = proj.split(d, dim=-1)
    return Q, K, V

def _self_check():
    B,T,d_model,d = 2,3,12,4
    X = torch.randn(B,T,d_model)
    W = torch.randn(3*d, d_model)
    b = torch.randn(3*d)
    try:
        Q,K,V = fused_qkv(X,W,b,d)
    except NotImplementedError:
        print("🔧  Implement fused_qkv.")
        return
    refQ,refK,refV = _ref_fused(X,W,b,d)
    assert torch.allclose(Q,refQ) and torch.allclose(K,refK) and torch.allclose(V,refV)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 48 – batched_index_select
================================
Given a 3-D tensor **seq** (B, T, D) and index tensor **idx** (B, K)
select row-specific timesteps, returning (B, K, D).

This is similar to `torch.gather` but idx is 2-D (no feature dim).

In [6]:
import torch

def batched_index_select(seq: torch.Tensor,
                         idx: torch.Tensor) -> torch.Tensor:
    # seq: B, T, D
    # idx: B, K
    B, T, D = seq.shape
    B, K = idx.shape
    return torch.gather(seq, 1, idx.view(B, K, 1).expand(B, K, D))


def _ref_bis(seq, idx):
    B,K = idx.shape
    D = seq.size(-1)
    idx_exp = idx.unsqueeze(-1).expand(-1,-1,D)
    return seq.gather(1, idx_exp)

def _self_check():
    B,T,D,K = 3,6,5,4
    seq = torch.randn(B,T,D)
    idx = torch.randint(0,T,(B,K))
    try:
        out = batched_index_select(seq, idx)
    except NotImplementedError:
        print("🔧  Implement batched_index_select.")
        return
    assert torch.allclose(out, _ref_bis(seq,idx))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 49 – Clip gradients by global L2-norm
============================================
Given a list of parameter tensors **params** (with `.grad`)
implement

        scale = clip / max(clip, global_norm)
        p.grad.mul_(scale)

Return the computed `global_norm`.

Ignore parameters whose `.grad` is None; use FP32 for accum. No loops
over tensor entries (loops over params allowed).

In [17]:
import torch
from typing import List

def clip_grad_global_norm(params: List[torch.Tensor],
                          clip: float) -> float:
    global_norm = sum([p.grad.pow(2).sum() for p in params if p.grad is not None])**0.5
    scale = clip / max(clip, global_norm)
    for p in params:
        if p.grad is not None:
            p.grad.mul_(scale)
    return global_norm

# reference + test
def _ref_clip(params, clip):
    total = 0.
    for p in params:
        if p.grad is not None:
            total += p.grad.pow(2).sum().item()
    gnorm = total ** 0.5
    scale = clip / max(clip, gnorm)
    for p in params:
        if p.grad is not None:
            p.grad.mul_(scale)
    return gnorm

def _get_params():
    torch.manual_seed(0)
    p1 = torch.randn(3,4, requires_grad=True)
    p2 = torch.randn(5, requires_grad=True)
    loss = (p1**2).sum() + (p2**2).sum()
    loss.backward()
    params = [p1,p2]
    return params

def _self_check():
    try:
        g = clip_grad_global_norm(_get_params(), 1.0)
    except NotImplementedError:
        print("🔧  Implement clip_grad_global_norm.")
        return
    ref = _ref_clip(_get_params(), 1.0)
    assert abs(g - ref) < 1e-6
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 50 – Prefix mean
=======================
Given x (B, T) compute y[i] = mean(x[0:i+1]) for each position
(cumulative average).  Vectorise without explicit loops.

In [14]:
import torch

def prefix_mean(x: torch.Tensor) -> torch.Tensor:
    B, T = x.shape
    y = torch.cumsum(x, dim=-1)
    z = (torch.arange(T) + 1).view(1, T)
    return y / z


def _ref_prefix(x):
    cumsum = x.cumsum(1)
    idx = torch.arange(1, x.size(1)+1, device=x.device)
    return cumsum / idx

def _self_check():
    B,T=2,5
    x = torch.randn(B,T)
    try:
        y = prefix_mean(x)
    except NotImplementedError:
        print("🔧  Implement prefix_mean.")
        return
    assert torch.allclose(y, _ref_prefix(x))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 51 – Nucleus (top-p) filter
==================================
Keep the *smallest* set of tokens whose cumulative soft-max probability
≥ **p** (0<p≤1) **per row**, set others to −∞.

Inputs
------
* logits : (B, V)
* p      : float threshold

Return filtered logits. No loops over V.

In [23]:
import torch

def nucleus_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    B, V = logits.shape
    probs = torch.softmax(logits, dim=-1)
    values, indices = torch.sort(probs, dim=-1, descending=True)
    mask = torch.cumsum(values, dim=-1) <= p
    mask[..., 1:] |= mask[..., :-1]
    _, remap = torch.sort(indices, dim=-1)
    re_mask = torch.gather(mask, -1, remap)
    logits.masked_fill_(~re_mask, -torch.inf)
    return logits
    


# reference + test
def _ref_nucleus(L, p):
    probs = torch.softmax(L, -1)
    sorted_probs, sorted_idx = probs.sort(descending=True)
    cum = sorted_probs.cumsum(-1)
    mask = cum <= p
    # keep also the first token that exceeds p
    mask[...,1:] |= mask[...,:-1]
    keep_idx = sorted_idx.masked_select(mask)
    out = L.new_full(L.shape, float("-inf"))
    out.scatter_(1, keep_idx.view(L.size(0), -1), L.gather(1, keep_idx.view(L.size(0), -1)))
    return out

def _self_check():
    torch.manual_seed(0)
    L = torch.randn(3,12)
    p = 0.9
    try:
        out = nucleus_filter(L, p)
    except NotImplementedError:
        print("🔧  Implement nucleus_filter.")
        return
    ref = _ref_nucleus(L,p)
    assert torch.allclose(torch.softmax(out, -1), torch.softmax(ref,-1), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 52 – Beam-search text decoder
=====================================
You are given a *batch* of autoregressive LM log-probability functions:

      logits_t = model(prefix)          # returns (B, V) **LOGITS**

Implement

      tokens, logp = beam_decode(
          model_fn,        # callable(step_tokens[B,L]) → logits[B,V]
          B,               # batch size
          beam_size=4,
          max_len=30,
          eos_id=2,
          length_penalty=0.6,
          topk_per_step=50,
          device="cpu"
      )

* **Beam search** keeps the `beam_size` best partial hypotheses
  (log-prob *plus* length-penalty) **per batch element**.
* At each step:
    1. Call `model_fn` with *all* current beams (flattened into one big
       `(B·beam, L)` tensor) and receive logits.
    2. Apply **top-k filtering** (keep the 50 highest logits per row).
    3. Expand each beam by 1 token; score = cumulative log-prob / LP,
       where LP = ((5+len)**length_penalty)/(6**length_penalty).
    4. Select the `beam_size` best new beams (finished-EOS stay alive).
* Stop when **all** beams in a batch element terminated with EOS **or**
  `max_len` reached.  Pad unfinished sequences with EOS at the end.
* Return:
    • **tokens** : (B, beam_size, L≤max_len)  best beams ordered by score
    • **logp**   : (B, beam_size)             final *length-penalised* score

No Python loops over the vocabulary V.  A loop over time steps is OK.

Hints
-----
* Keep batched data structures: `alive_logp`, `alive_tokens`.
* `torch.topk` and `flatten()` will be your friends.

In [ ]:
import torch
from typing import Callable, Tuple

def beam_decode(model_fn:      Callable[[torch.Tensor], torch.Tensor],
                batch_size:    int,
                vocab_size:    int,
                beam_size:     int = 4,
                max_len:       int = 30,
                eos_id:        int = 2,
                length_penalty:float = 0.6,
                topk_per_step: int = 50,
                device:        str = "cpu"
               ) -> Tuple[torch.Tensor, torch.Tensor]:
    #
    # model_fn: (B, M, L) -> (B, M, V)
    #
    B, V, M, k, alpha = batch_size, vocab_size, beam_size, topk_per_step, length_penalty
    
    beam_tokens = torch.full((B, M, 1), eos_id)
    beam_logp = torch.zeros(B, M)
    
    for L in range(1, max_len+1):
        logit = model_fn(beam_tokens) # (B, M, L) -> (B, M, V)
        logp = torch.log_softmax(logit, dim=-1)
        # force eos
        finished = torch.sum(beam_tokens == eos_id, dim=-1) > 1    # B, M
        logp[finished] = -torch.inf
        logp[finished][:, eos_id] = 0.0
        #
        topk_val, topk_idx = logp.topk(k=topk_per_step, largest=True) # B, M, k
        # expand beam
        exp_beam_tokens = torch.cat([beam_tokens.unsqueeze(-2).expand(-1, -1, k, -1), topk_idx.unsqueeze(-1)], dim=-1) # B, M, k, L+1
        exp_beam_logp = beam_logp.unsqueeze(-1) + topk_val # B, M, k
        exp_beam_tokens = exp_beam_tokens.view(B, M*k, -1) # B, Mk, L+1
        exp_beam_logp = exp_beam_logp.view(B, M*k)         # B, Mk
        # reduce beam
        score = exp_beam_logp / _lp(torch.sum(exp_beam_tokens != eos_id, dim=-1), alpha)
        _, top_idx = score.topk(k=M, largest=True) # B, M
        beam_tokens = exp_beam_tokens.gather(1, top_idx.view(B, M, 1).expand(-1, -1, L+1))
        beam_logp = exp_beam_logp.gather(1, top_idx)

    return beam_tokens, beam_logp
    
    
# ---------------------------------------------------------------------
# -------------------- reference & self-check -------------------------
def _lp(seq_len, alpha):
    return ((5+seq_len).float() / 6)**alpha

@torch.no_grad()
def _ref_beam_decode(model_fn, B, V, beam=4, max_len=30, eos=2,
                     alpha=0.6, step_topk=50, device="cpu"):
    alive_tok  = torch.full((B, beam, 1), eos, device=device, dtype=torch.long)
    alive_lp   = torch.zeros (B, beam, device=device)        # length-normed
    finished_tok = torch.full((B, 0, 1), eos, device=device, dtype=torch.long)
    finished_lp  = torch.full((B, 0       ), -1e9, device=device)

    for t in range(1, max_len+1):
        flat_tok  = alive_tok.view(B*beam, -1)
        logits    = model_fn(flat_tok)                       # (B*beam,V)
        logits    = logits.view(B, beam, V)
        topk_val, topk_idx = logits.topk(step_topk, -1)      # (B,beam,k)

        cand_lp = alive_lp.unsqueeze(-1) + topk_val          # (B,beam,k)
        cand_len = torch.full_like(cand_lp, t)
        cand_lp = cand_lp / _lp(cand_len, alpha)

        cand_tok = torch.cat([ alive_tok.unsqueeze(-1).expand(-1,-1,-1,step_topk),
                               topk_idx.unsqueeze(2) ], 2)   # (B,beam,k,L+1)

        Bc, beamc, kc, Lp1 = cand_tok.shape
        cand_tok = cand_tok.view(B, beam*kc, Lp1)
        cand_lp  = cand_lp .view(B, beam*kc)

        # finished?
        eos_mask = cand_tok[:,:, -1].eq(eos)
        finished_lp = torch.cat([finished_lp, cand_lp.masked_select(eos_mask)
                                 .view(B,-1)], 1)
        finished_tok = torch.cat([finished_tok,
                                   cand_tok[eos_mask].view(B,-1,Lp1)], 1)

        cand_lp.masked_fill_(eos_mask, -1e9)   # mask finished for alive
        new_lp, new_idx = cand_lp.topk(beam, 1)
        alive_lp  = new_lp
        gather_idx = new_idx.unsqueeze(-1).expand(-1,-1,Lp1)
        alive_tok = cand_tok.gather(1, gather_idx)

        all_done = (finished_lp.size(1) >= beam) | (t==max_len)
        if all_done.all(): break

    # final merge
    need   = beam - finished_lp.size(1)
    if need > 0:
        finished_lp  = torch.cat([finished_lp, alive_lp[:,:need]], 1)
        finished_tok = torch.cat([finished_tok, alive_tok [:,:need]], 1)

    order = finished_lp.argsort(dim=1, descending=True)
    B_idx = torch.arange(B).unsqueeze(1)
    final_tok = finished_tok[B_idx, order]
    final_lp  = finished_lp [B_idx, order]
    return final_tok, final_lp

# Dummy model: next token logits ~ N(0,1)
class _DummyLM:
    def __init__(self, V=100):
        self.V = V
    def __call__(self, prefix):
        prefix_shape = prefix.shape
        logits = torch.randn(prefix_shape[:-1] + (self.V,), device=prefix.device)
        logits[..., 0] = -1e9      # mask token 0 as invalid
        return logits

def _self_check():
    B, V = 2, 50
    lm = _DummyLM(V=V)
    try:
        tok, lp = beam_decode(lm, B, V, beam_size=3, max_len=12, device="cpu")
    except NotImplementedError:
        print("🔧  Implement beam_decode.")
        return
    ref_tok, ref_lp = _ref_beam_decode(lm, B, V, beam=3, max_len=12)
    assert tok.shape == ref_tok.shape
    assert torch.allclose(lp, ref_lp, atol=1e-4)
    print("✅  Beam-search basic test passed")444

_self_check()

Problem 53 – INT8 per-channel linear layer
=========================================
Write `linear_int8(x, w_int8, scale_w, zp_w, scale_x, zp_x)` that
performs **inference** for a fully-connected layer with *asymmetric*
affine quantisation:

    q_x = round(x / scale_x) + zp_x                 # int8
    q_w = w_int8   (already stored int8, per-OUT-channel scale)

De-quant-in-float algorithm
---------------------------
1. Compute int32 matmul:   y_int = (q_x  @  q_wᵀ)
2. De-quantise row i, col j:

    y[i,j] = (y_int[i,j]  -  zp_x·Σ q_w[j]  -  zp_w[j]·Σ q_x[i]
              + zp_x·zp_w[j]·D) * scale_x * scale_w[j]

where D = input dimension.

Inputs
------
* **x**        : (B, D)   float32 activations
* **w_int8**   : (O, D)   int8   weights
* **scale_w**  : (O,)     float32, per-out-channel
* **zp_w**     : (O,)     int8    zero-points per-out
* **scale_x**  : scalar
* **zp_x**     : int8
Return **y** (B, O) float32.  No loops over B, D, O.

In [43]:
import torch
from typing import Tuple

def linear_int8(x: torch.Tensor,
                qw: torch.Tensor,
                sw: torch.Tensor,
                zpw: torch.Tensor,
                sx: float,
                zpx: int
               ) -> torch.Tensor:
    B, D = x.shape
    O, _ = qw.shape
    
    sx = torch.full((B, 1), sx, dtype=torch.float)
    zpx = torch.full((B, 1), zpx, dtype=torch.int)
    sw = sw.view(1, O)
    zpw = zpw.view(1, O)
    
    qx = torch.round(x / sx).clamp(-128, 127).int() + zpx # qx: B, D
                                                          # qw: O, D
    
    qy = qx.int() @ qw.T.int() # B, O
    sum_qx = qx.sum(dim=-1).view(B, 1)
    sum_qw = qw.sum(dim=-1).view(1, O)
    
    y = (qy - sum_qx * zpw - zpx * sum_qw + D * zpx * zpw) * (sx * sw)
    return y
    
    
    
# -----------------------------------------------------------------
def _ref_linear_int8(x, wq, sw, zpw, sx, zpx):
    B,D = x.shape
    O,_ = wq.shape
    qx  = ((x / sx).round() + zpx).clamp(-128,127).int()   # (B,D)
    wq64= wq.int()
    y_int = qx @ wq64.T                                              # (B,O)

    sum_qw = wq64.sum(-1)            # (O,)
    sum_qx = qx .sum(-1)             # (B,)
    
    term = y_int - zpx*sum_qw - zpw*sum_qx.unsqueeze(1) + D*zpx*zpw.int()
    y = term * sx * sw
    return y

def _self_check():
    torch.manual_seed(0)
    B,D,O = 6, 32, 8
    x = torch.randn(B,D)
    w = torch.randint(-120, 120, (O,D), dtype=torch.int8)
    sw = torch.rand(O) * 0.02
    zpw= torch.randint(-3,3,(O,), dtype=torch.int8)
    sx = 0.03
    zpx= -5
    try:
        y = linear_int8(x,w,sw,zpw,sx,zpx)
    except NotImplementedError:
        print("🔧  Implement linear_int8.")
        return
    ref = _ref_linear_int8(x,w,sw,zpw,sx,zpx)
    assert torch.allclose(y, ref, atol=1e-3)
    print("✅  INT8 linear forward passed")

_self_check()

✅  INT8 linear forward passed


Problem 54 – Viterbi best-path CTC decode
========================================
Given **log-probs** from a CTC network:

    logp : (T, V)  (natural-log, already softmaxed) 

and *blank id* `blank`, return the **most-probable output sequence**
after collapsing repeats and removing blanks.

Constraints
-----------
* Non-greedy: you must compute the *exact* Viterbi path (DP, log-space).
* Return both (best path tokens as python list, log-prob of that path).
* No explicit loops over V (loops over T OK).  Use fancy indexing.

In [ ]:
import torch
from typing import List, Tuple

def ctc_viterbi(logp: torch.Tensor, blank: int = 0
               ) -> Tuple[List[int], float]:
    best_v, best_path = logp.max(dim=-1)
    best_v = best_v.tolist()
    best_path = best_path.tolist()
    out = [v for i,v in enumerate(best_path) if v!=blank and (i==0 or v!=best_path[i-1])]
    return out, sum(best_v)
    
# -----------------------------------------------------------------
def _ref_viterbi(logp, blank=0):
    T,V = logp.shape
    prev = torch.full((V,), float("-inf"), device=logp.device)
    prev[blank] = logp[0,blank]
    path = [[blank] for _ in range(V)]
    for t in range(1,T):
        cur = torch.full_like(prev, float("-inf"))
        new_path = [None]*V
        for v in range(V):
            stay  = prev[v]
            trans = prev.argmax()     # from any symbol (slow but fine for ref)
            best_prev_logp = max(stay, prev[trans])
            if best_prev_logp == stay:
                new_path[v] = path[v] + [v]
            else:
                new_path[v] = path[trans] + [v]
            cur[v] = best_prev_logp + logp[t,v]
        prev, path = cur, new_path
    best_v = prev.argmax().item()
    best_path = path[best_v]
    # collapse
    out = [v for i,v in enumerate(best_path)
             if v!=blank and (i==0 or v!=best_path[i-1])]
    return out, prev[best_v].item()

def _self_check():
    torch.manual_seed(0)
    T,V=12,6
    lp = torch.log_softmax(torch.randn(T,V), -1)
    try:
        seq,score = ctc_viterbi(lp, blank=0)
    except NotImplementedError:
        print("🔧  Implement ctc_viterbi.")
        return
    rseq,rscore = _ref_viterbi(lp,0)
    print(seq, rseq, score, rscore)
    assert seq==rseq and abs(score-rscore)<1e-3
    print("✅  Viterbi CTC basic passed")

_self_check()

Problem 55 – Flash-Attention forward (single head)
=================================================
Compute causal soft-max attention **without** instantiating the full
(T,T) score matrix.

Algorithm (H ≡ hidden-dim):
---------------------------
Process queries/keys/values in BLOCKS of rows/cols size *Bsz*:

for q_block in range(0,T,Bsz):
    1. Initialise `m = -inf`, `l = 0`, `out = 0`
    2. For k_block in 0 … q_block  (causal):
         • score = Q_q @ K_kᵀ / √H           # (bq,bk)
         • mask future cols if k_block==q_block
         • m_new = max(m, score.max(-1))
         • l    *= exp(m - m_new)
           l    += exp(score - m_new).sum(-1)
         • out  *= exp(m - m_new).unsqueeze(-1)
           • soft = exp(score - m_new) / l.unsqueeze(-1)
           • out  += soft @ V_k
         • m = m_new
Return concatenated `out` (B,T,H).

Implement **flash_attn(Q,K,V,block)`** exactly as above, with
  *Q,K,V* = (T,H) tensors, block size parameter.

No python loops over hidden-dim H; outer loops over blocks OK.

In [54]:
import torch, math

def flash_attn(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, block: int = 128) -> torch.Tensor:
    T, D = Q.shape
    scale = 1/math.sqrt(D)
    O_chunks = []
    for qs in range(0, T, block):
        qe = min(qs+block, T)

        m = torch.full((qe-qs,), float("-inf"))   # max score
        l = torch.zeros_like(m)                   # bottom
        o = torch.zeros((qe-qs, D))               # top
        for ks in range(0, qe, block):
            ke = min(ks+block, qe)
            S = (Q[qs:qe] @ K[ks:ke].T) * scale
            if ks == qs:
                mask = torch.tril(torch.ones_like(S, dtype=torch.bool), diagonal=0)
                S.masked_fill_(~mask, -torch.inf)
            m_new = torch.maximum(m, S.max(-1).values)
            l = torch.exp(m-m_new)*l + torch.exp(S-m_new.unsqueeze(-1)).sum(-1)
            o = torch.exp(m-m_new).unsqueeze(-1)*o + torch.exp(S-m_new.unsqueeze(-1)) @ V[ks:ke]
            m = m_new
        O_chunks.append(o / l.unsqueeze(-1))
    return torch.cat(O_chunks, 0)

# -----------------------------------------------------------------
def _ref_flash(Q,K,V,block):
    T,H = Q.shape
    scale = 1/math.sqrt(H)
    out_chunks=[]
    for qs in range(0,T,block):
        qe=min(qs+block,T)
        Qb = Q[qs:qe]
        m = torch.full((qe-qs,), float("-inf"), device=Q.device)
        l = torch.zeros_like(m)
        ob= torch.zeros((qe-qs,H), device=Q.device)
        for ks in range(0,qe,block):
            ke=min(ks+block,qe)
            Sb = (Qb @ K[ks:ke].T)*scale
            if ks==qs:
                tri_mask = torch.triu(torch.ones_like(Sb), diagonal=1).bool()
                Sb.masked_fill_(tri_mask, float("-inf"))
            m_new = torch.maximum(m, Sb.max(-1).values)
            l = torch.exp(m-m_new)*l + torch.exp(Sb-m_new.unsqueeze(-1)).sum(-1)
            ob = ob*torch.exp((m-m_new).unsqueeze(-1)) \
               + (torch.exp(Sb-m_new.unsqueeze(-1)) @ V[ks:ke])
            m = m_new
        out_chunks.append(ob/l.unsqueeze(-1))
    return torch.cat(out_chunks,0)

def _self_check():
    torch.manual_seed(0)
    T,H=64,32
    Q= torch.randn(T,H); K= torch.randn(T,H); V= torch.randn(T,H)
    try:
        out = flash_attn(Q,K,V,block=16)
    except NotImplementedError:
        print("🔧  Implement flash_attn.")
        return
    ref = _ref_flash(Q,K,V,16)
    assert torch.allclose(out, ref, atol=1e-4)
    print("✅  Flash-attention forward passed")

_self_check()

✅  Flash-attention forward passed


Problem 57 – Vectorised greedy decoding with dynamic batch-trim
==============================================================
Implement a *token*-by-token greedy decoder that stops calling the model
for any sequence once it has emitted `eos_id`.

Signature
---------
    tokens, lens = greedy_decode(
        model_fn,          # callable: logits = model_fn(prefix)  – (B,V)
        max_len=64,
        eos_id=2,
        pad_id=0,
        device="cpu"
    )

* **model_fn(prefix)** receives **all *alive* sequences only** as an
  int64 tensor `(B_alive, L)` and returns logits `(B_alive, V)` for the
  next step.  (The prefix *lengths* differ across batch elements.)
* Begin every sequence with a single BOS token = `eos_id`.
* At each step
    1. Call `model_fn` *only* on rows that are **not yet finished**  
       (use a boolean mask to skip finished rows – **no Python loops
       over the batch**).
    2. Greedily select `argmax` token.
    3. Update `finished` mask  (when token == eos_id).
* **Pre-allocate** a `(B, max_len)` result tensor filled with `pad_id`
  and write tokens in-place.  
  Stop early when *all* sequences are finished **or** `max_len`
  reached.

Return
------
* **tokens** : (B, max_len)  padded with `pad_id`
* **lens**   : (B,)          actual decoded lengths *including* EOS

In [60]:
import torch
from typing import Callable, Tuple

def greedy_decode(model_fn : Callable[[torch.Tensor], torch.Tensor],
                  B        : int,
                  V        : int,
                  max_len  : int = 64,
                  eos_id   : int = 2,
                  pad_id   : int = 0,
                  device   : str = "cpu"
                 ) -> Tuple[torch.Tensor, torch.Tensor]:
    tok = torch.full((B, max_len), pad_id, dtype=torch.long)
    tok[:, 0] = eos_id
    lens = torch.full((B,), 1, dtype=torch.long)
    finished = torch.zeros(B, dtype=torch.bool)
    for t in range(1, max_len):
        if torch.all(finished):
            break
        logit = model_fn(tok[~finished, :t])
        next_tk = logit.argmax(-1)
        tok[~finished, t] = next_tk
        lens[~finished] = t+1
        finished = finished | (next_tk == eos_id)
    return tok, lens
# -------------------------------------------------------------------
# ------------------ reference + mini-self-check --------------------
@torch.no_grad()
def _ref_greedy(model_fn,B,V,max_len=64,eos=2,pad=0,device="cpu"):
    tok = torch.full((B, max_len), pad, dtype=torch.long, device=device)
    lens= torch.full((B,),       1,   dtype=torch.long, device=device) # BOS
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    alive_idx = torch.arange(B, device=device)
    step_inp  = torch.full((B,1), eos, dtype=torch.long, device=device)
    tok[:,0]  = eos
    for t in range(1, max_len):
        if finished.all(): break
        logits = model_fn(step_inp[~finished])          # (B_alive,V)
        next_tk= logits.argmax(-1)                      # (B_alive,)
        tok[~finished, t] = next_tk
        step_inp[~finished, 0] = next_tk
        lens[~finished] = t+1
        finished |= next_tk.eq(eos)
    return tok, lens

class _ToyLM:
    """logits = one-hot cyclical counter mod V"""
    def __init__(self,V):
        self.V=V
    def __call__(self,prefix):
        nxt = (prefix[:,-1]+1) % self.V
        return torch.nn.functional.one_hot(nxt, self.V).float()*1e6

def _self_check():
    torch.manual_seed(0)
    B,V=4,11
    lm=_ToyLM(V)
    try:
        t,l = greedy_decode(lm,B,V,max_len=10,device="cpu")
    except NotImplementedError:
        print("🔧  Implement greedy_decode.")
        return
    rt,rl=_ref_greedy(lm,B,V,max_len=10)
    assert torch.equal(t,rt) and torch.equal(l,rl)
    print("✅  Greedy dynamic-batch decoder passed")

_self_check()

✅  Greedy dynamic-batch decoder passed


Problem 58 – Memory-efficient top-k cosine similarity
====================================================
Given two *huge* L2-normalised embedding banks **A** (N,D) and **B**
(M,D), return for every row of A the **indices** of the `k` most
similar rows in B **without** materialising the full (N×M) matrix.

Blocked algorithm
-----------------
Process B in column-blocks of size *Bsz*:

    init best_val  = −1, best_idx = −1
    for start in 0 … M step Bsz:
        end  = min(start+Bsz, M)
        cos  = A  @  B[start:end].T                # (N, Bsz)
        update running top-k (values & indices)

You may use `torch.topk` on the *(N, Bsz)* chunks.  Do **not**
write Python loops over N.

Return `indices` of shape *(N, k)* sorted **by descending cosine** per
row (ties arbitrary).

Try to keep intermediate peak memory ≪ N×M.

In [46]:
import torch
from typing import Tuple

def topk_cosine_block(A: torch.Tensor,
                      B: torch.Tensor,
                      k: int = 10,
                      block: int = 16384
                     ) -> torch.Tensor:
    """
    A : (N,D)  L2-normalised
    B : (M,D)  L2-normalised
    """
    N, D = A.shape
    M, _ = B.shape
    best_val = -torch.ones(N, k)
    best_idx = -torch.ones(N, k, dtype=torch.int)
    for ss in range(0, M, block):
        ee = min(ss + block, M)
        BB = B[ss:ee, :]
        V = torch.einsum("nd,md->nm", A, BB)
        merged_val = torch.cat([best_val, V], dim=-1)
        merged_idx = torch.cat([best_idx, torch.arange(ss, ee, dtype=torch.int).view(1, ee-ss).expand(N, ee-ss)], dim=-1)
        
        best_val, bid = torch.topk(merged_val, k, dim=-1, largest=True, sorted=True) # N, k
        best_idx = torch.gather(merged_idx, -1, bid)

    return best_idx
        
    
# -------------------------------------------------------------------
# ------------------ reference + mini-self-check --------------------
def _ref_topk_block(A,B,k=10,block=16384):
    N,D=A.shape
    best_val = torch.full((N,k), -1., device=A.device)
    best_idx = torch.full((N,k), -1 , device=A.device, dtype=torch.long)
    for s in range(0,B.size(0),block):
        e=min(s+block, B.size(0))
        cos = A @ B[s:e].T                         # (N, bsz)
        merged = torch.cat([best_val, cos], 1)
        merged_idx = torch.cat([best_idx,
                                torch.arange(s,e, device=A.device)
                                      .repeat(N,1)], 1)
        best_val, topk_idx = merged.topk(k, 1)     # values
        best_idx = merged_idx.gather(1, topk_idx)
    order = best_val.argsort(1, descending=True)
    return best_idx.gather(1, order)

def _self_check():
    torch.manual_seed(0)
    N,M,D,k = 500, 1800, 64, 7
    A = torch.randn(N,D); B=torch.randn(M,D)
    A = torch.nn.functional.normalize(A, dim=1)
    B = torch.nn.functional.normalize(B, dim=1)
    try:
        idx = topk_cosine_block(A,B,k,block=256)
    except NotImplementedError:
        print("🔧  Implement topk_cosine_block.")
        return
    ridx=_ref_topk_block(A,B,k,block=256)
    assert torch.equal(idx, ridx)
    print("✅  Block-top-k cosine search passed")

_self_check()

✅  Block-top-k cosine search passed


Problem 59 – Layer Normalization: custom forward & backward
===========================================================
Implement LayerNorm over the **last** dimension for inputs X ∈ ℝ^{N×D}
with learnable (γ, β) ∈ ℝ^D. Use a custom autograd Function.

Definitions
-----------
Let
    μ  = mean(X, dim=-1, keepdim=True)                # (N,1)
    σ² = mean((X − μ)², dim=-1, keepdim=True)         # (N,1)
    inv_std = 1 / sqrt(σ² + eps)                      

    X̂  = (X − μ) * inv_std                            # (N,D)
    Y   = γ ⊙ X̂ + β                                   # (N,D)

You must implement both forward and backward.

Gradients
---------
Given upstream gradient dY:
  Let D be the last-dim size and define:

    dβ      = Σ_{n} dY_n                               # sum over batch N
    dγ      = Σ_{n} (dY_n ⊙ X̂_n)

    dX = (inv_std / D) ⊙ γ ⊙ ( D·dY 
          − 1·Σ(dY,dim=-1,keepdim=True)
          − X̂ ⊙ Σ(dY ⊙ X̂, dim=-1, keepdim=True) )

Return dX, dγ, dβ.

Task
----
Fill in both `forward` and `backward` of `LayerNormFn` below. Then
`LayerNorm` (nn.Module) wraps it to match nn.LayerNorm on (N,D).

The self-check compares values and gradients against
`torch.nn.functional.layer_norm`.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

# -------------------------- YOUR CODE -----------------------------
class LayerNormFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                X: torch.Tensor,         # (B,D)
                gamma: torch.Tensor,     # (D,)
                beta: torch.Tensor,      # (D,)
                eps: float) -> torch.Tensor:
        B, D = X.shape

        mu = X.sum(dim=-1, keepdim=True) / D
        Xbar = X - mu
        var = Xbar.pow(2).sum(dim=-1, keepdim=True) / D
        eta = (var + eps).pow(-0.5)
        Xhat = Xbar * eta
        Y = Xhat * gamma + beta
        
        ctx.save_for_backward(X, gamma, beta, mu, Xbar, var, eta, Xhat, Y)
        ctx.eps = eps

        return Y
        
    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, None]:
        X, gamma, beta, mu, Xbar, var, eta, Xhat, Y = ctx.saved_tensors
        eps = ctx.eps
        
        B, D = X.shape
        
        grad_gamma = (Xhat * grad_out).sum(dim=0)
        grad_beta = grad_out.sum(dim=0)
        grad_Xhat = grad_out * gamma
        
        grad_eta = (Xbar * grad_Xhat).sum(dim=-1, keepdim=True)
        grad_var = -0.5 * (var + eps).pow(-1.5) * grad_eta
        grad_Xbar = grad_Xhat * eta + 2 * Xbar * grad_var / D

        grad_mu = - grad_Xbar.sum(dim=-1, keepdim=True)
        grad_X = grad_Xbar + grad_mu / D
        
        return grad_X, grad_gamma, grad_beta, None

class LayerNorm(nn.Module):
    def __init__(self, D: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(D))
        self.bias   = nn.Parameter(torch.zeros(D))
        self.eps    = eps

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return LayerNormFn.apply(X, self.weight, self.bias, self.eps)
# ------------------------------------------------------------------


# -------------------- Reference + Self-check ----------------------
class _RefLN(nn.Module):
    def __init__(self, D, eps=1e-5):
        super().__init__()
        self.w = nn.Parameter(torch.ones(D))
        self.b = nn.Parameter(torch.zeros(D))
        self.eps = eps
    def forward(self, X):
        return F.layer_norm(X, (X.size(-1),), self.w, self.b, self.eps)
#         return _ref_formula(X, self.w, self.b, self.eps)

def _ref_formula(X, gamma, beta, eps):
    mu = X.mean(-1, keepdim=True)
    var = ((X - mu)**2).mean(-1, keepdim=True)
    inv_std = torch.rsqrt(var + eps)
    X_hat = (X - mu) * inv_std
    return gamma * X_hat + beta

def _self_check():
    torch.manual_seed(0)
    N, D = 2, 3
    X = torch.randn(N, D, requires_grad=True)
    ln_ref = _RefLN(D, eps=1e-5)
    ln_me  = LayerNorm(D, eps=1e-5)
    # align parameters
    with torch.no_grad():
        ln_me.weight.copy_(ln_ref.w)
        ln_me.bias.copy_(ln_ref.b)

    try:
        Y_me = ln_me(X)
    except NotImplementedError:
        print("🔧  Implement LayerNormFn.forward/backward.")
        return

    Y_ref = ln_ref(X)

    # Forward equality
    assert torch.allclose(Y_me, Y_ref, atol=1e-6), "Forward mismatch"

    # Backward equality (w.r.t X, gamma, beta)
    g = torch.randn_like(Y_me)
    Y_ref.backward(g, retain_graph=True)
    dX_ref, dw_ref, db_ref = X.grad.clone(), ln_ref.w.grad.clone(), ln_ref.b.grad.clone()

    X.grad.zero_(); ln_ref.w.grad.zero_(); ln_ref.b.grad.zero_()
    Y_me.backward(g)
    dX_me, dw_me, db_me = X.grad, ln_me.weight.grad, ln_me.bias.grad

    assert torch.allclose(dX_me, dX_ref, atol=1e-6), "dX mismatch"
    assert torch.allclose(dw_me, dw_ref, atol=1e-6), "dγ mismatch"
    assert torch.allclose(db_me, db_ref, atol=1e-6), "dβ mismatch"

    print("✅  LayerNorm forward & backward passed")

_self_check()

✅  LayerNorm forward & backward passed


Problem 60 – Decoding with top-k and top-p sampling
==================================================
Write a *vectorised* autoregressive sampler that combines **top-k**
and **top-p** filtering at each step, then samples one token.

Signature
---------
tokens, lengths = sample_decode(
    model_fn,      # callable(prefix[B_alive, L]) → logits[B_alive, V]
    B, V, max_len,
    eos_id=2, pad_id=0,
    top_k=50, top_p=0.9, temperature=1.0, device="cpu"
)

Rules
-----
• Start each sequence with BOS = eos_id at position 0.

• At each step t:

    1) Call model_fn on *only* unfinished rows.  
    2) Divide logits by `temperature`.  
    3) **Top-k**: keep k highest logits per row, others → −∞.  
    4) **Top-p**: among remaining, keep the *smallest* set whose
       cumulative probability ≥ p (include the first above-p token);
       zero out the rest; renormalise.  
    5) Sample with `torch.multinomial` (no Python loops over batch).  
    6) Stop rows that emit EOS; end early if all rows finished or
       `max_len` reached.

• Pre-allocate `(B, max_len)` output padded with `pad_id`; lengths
  include the EOS position.

Return
------
tokens : (B, max_len),  lengths : (B,)

In [4]:
import torch
import torch.nn.functional as F
from typing import Callable, Tuple

# -------------------------- YOUR CODE -----------------------------
def sample_decode(model_fn : Callable[[torch.Tensor], torch.Tensor],
                  B        : int,
                  V        : int,
                  max_len  : int,
                  eos_id   : int = 2,
                  pad_id   : int = 0,
                  top_k    : int = 50,
                  top_p    : float = 0.9,
                  temperature: float = 1.0,
                  device   : str = "cpu"
                 ) -> Tuple[torch.Tensor, torch.Tensor]:
    raise NotImplementedError
# ------------------------------------------------------------------


# -------------------- Reference + Self-check ----------------------
@torch.no_grad()
def _ref_sample(model_fn,B,V,max_len,eos=2,pad=0,k=50,p=0.9,temp=1.0,device="cpu"):
    toks = torch.full((B,max_len), pad, dtype=torch.long, device=device)
    lens = torch.ones(B, dtype=torch.long, device=device)  # BOS counted
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    prev = torch.full((B,1), eos, dtype=torch.long, device=device)
    toks[:,0] = eos

    for t in range(1, max_len):
        if finished.all(): break
        logits = model_fn(prev[~finished]) / max(temp, 1e-8)  # (B_alive,V)

        # top-k
        if k < V:
            val, idx = logits.topk(k, dim=-1)
            keep = torch.full_like(logits, float("-inf"))
            keep.scatter_(1, idx, logits.gather(1, idx))
            logits = keep

        # top-p
        probs = F.softmax(logits, dim=-1)
        sorted_p, sorted_idx = probs.sort(dim=-1, descending=True)
        cum = sorted_p.cumsum(-1)
        keep_mask = cum <= p
        # ensure at least one token kept
        keep_mask[..., 0] = True
        # map mask back to vocab order
        unsort_mask = torch.zeros_like(probs, dtype=torch.bool)
        unsort_mask.scatter_(1, sorted_idx, keep_mask)
        probs = probs * unsort_mask
        probs = probs / probs.sum(-1, keepdim=True).clamp_min(1e-12)

        next_tok = torch.multinomial(probs, 1).squeeze(-1)   # (B_alive,)
        toks[~finished, t] = next_tok
        prev[~finished, 0] = next_tok
        lens[~finished] = t+1
        finished |= next_tok.eq(eos)

    return toks, lens

# Toy LM: advances mod V deterministically; temperature irrelevant
class _ToyLM:
    def __init__(self, V): self.V = V
    def __call__(self, prefix):
        nxt = (prefix[:,-1] + 1) % self.V
        logits = F.one_hot(nxt, self.V).float() * 1e6
        return logits

def _self_check():
    torch.manual_seed(0)
    B,V,max_len=4,17,12
    lm = _ToyLM(V)
    try:
        t, l = sample_decode(lm, B, V, max_len, eos_id=2, pad_id=0, top_k=5, top_p=0.8, temperature=0.7)
    except NotImplementedError:
        print("🔧  Implement sample_decode.")
        return
    rt, rl = _ref_sample(lm, B, V, max_len, eos=2, pad=0, k=5, p=0.8, temp=0.7)
    assert torch.equal(t, rt) and torch.equal(l, rl)
    print("✅  Top-k + Top-p decoding passed")

_self_check()

🔧  Implement sample_decode.


Problem 61 – SGD optimizer from scratch (PyTorch-compatible step)
================================================================
Write a minimal **SGD** optimizer that matches `torch.optim.SGD`
for the following features:

* learning rate `lr`
* `momentum` (0 allowed)
* `weight_decay` (L2)
* `dampening`
* `nesterov` (requires momentum>0 and dampening==0)

Update Rule
-----------
For a parameter tensor **w**, gradient **g**:

    if weight_decay: g ← g + weight_decay · w
    if momentum:
        buf ← momentum · buf + (1 − dampening) · g      (per-parameter buffer)
        step_dir = g + momentum · buf    (if nesterov) 
                 = buf                   (else)
    else:
        step_dir = g

    w ← w − lr · step_dir

Task
----
Fill in `CustomSGD.step()` to perform the in-place update. The self-check
compares trajectories to PyTorch’s SGD on a small regression problem.

In [ ]:
import torch
import torch.nn.functional as F
from typing import List, Dict, Any

# -------------------------- YOUR CODE -----------------------------
class CustomSGD:
    def __init__(self, params: List[torch.Tensor],
                 lr: float = 1e-2,
                 momentum: float = 0.0,
                 weight_decay: float = 0.0,
                 dampening: float = 0.0,
                 nesterov: bool = False):
        if nesterov and (momentum <= 0 or dampening != 0.0):
            raise ValueError("Nesterov requires momentum>0 and dampening==0")
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay
        self.dampening = dampening
        self.nesterov = nesterov
        self.state: List[Dict[str, Any]] = [{"buf": None} for _ in self.params]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    @torch.no_grad()
    def step(self):
        raise NotImplementedError   # <-- implement in-place SGD step
# ------------------------------------------------------------------


# ---------------------- Reference + Self-check --------------------
def _self_check_sgd():
    torch.manual_seed(0)
    N, D = 64, 5
    X = torch.randn(N, D)
    true_w = torch.randn(D)
    y = X @ true_w + 0.1 * torch.randn(N)

    w_ref = torch.randn(D, requires_grad=True)
    w_me  = w_ref.clone().detach().requires_grad_()

    opt_ref = torch.optim.SGD([w_ref], lr=5e-2, momentum=0.9, weight_decay=1e-3,
                              dampening=0.0, nesterov=True)
    opt_me  = CustomSGD([w_me], lr=5e-2, momentum=0.9, weight_decay=1e-3,
                        dampening=0.0, nesterov=True)
    try:
        for _ in range(50):
            # reference
            opt_ref.zero_grad()
            loss_ref = 0.5 * ((X @ w_ref - y) ** 2).mean()
            loss_ref.backward()
            opt_ref.step()
            # ours
            opt_me.zero_grad()
            loss_me = 0.5 * ((X @ w_me - y) ** 2).mean()
            loss_me.backward()
            opt_me.step()
    except NotImplementedError:
        print("🔧  Implement CustomSGD.step().")
        return

    assert torch.allclose(w_me, w_ref, atol=1e-6), f"Param mismatch (max |Δ|={ (w_me-w_ref).abs().max():.2e })"
    print("✅  CustomSGD matches torch.optim.SGD trajectory")

_self_check_sgd()